# Edge AI: Knowledge Distillation + Hardware-Aware NAS (Colab, T4)

One notebook for the whole assignment:

* **Q1:** ResNet-50 teacher, distilled into MobileNetV2, SqueezeNet and EfficientNet-Lite0 on CIFAR-100. Uses soft targets with temperature, combined hard and soft loss, and feature distillation, with baselines trained without KD.
* **Q2:** Evolutionary, hardware-aware NAS on CIFAR-10. Candidates are rejected on measured CPU latency, ONNX size and peak memory, and the best one is retrained from scratch.
* **Deployment:** ONNX export, ONNX Runtime CPU inference, a benchmark CSV and the PDF technical report.

**How to run:** select *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
Total time is about **35–45 min** on a T4 (an estimate; not yet timed on a real T4). It uses the `colab` profile in `edgekd/config.py`. To make it shorter, lower the epoch counts in that cell.

| stage | ≈ time |
|---|---|
| teacher (ImageNet ResNet-50 fine-tuned, 3 ep @ 96 px) | 3–5 min |
| 3 baseline students (10 ep each) | 6–9 min |
| 3 KD students (10 ep each) | 10–14 min |
| NAS search (8 × 5 generations) + retrain best (15 ep) | 8–12 min |
| ONNX export, inference, benchmark, report | 3 min |

When it finishes, section 11 downloads **`edge_kd_nas_full.zip`**. It holds the complete project: every trained checkpoint (`.pt`), the ONNX models, results CSV/JSON, logs, the PDF report, figures and the source code.

In [ ]:
# 1. Check GPU + install the few packages Colab lacks
!nvidia-smi -L
!pip install -q onnx onnxruntime onnxscript reportlab
import torch; assert torch.cuda.is_available(), "Select a GPU runtime (Runtime > Change runtime type > T4 GPU)"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Source code
Each cell below writes one file of the `edgekd` package / scripts (identical to the GitHub repo).

In [ ]:
!mkdir -p /content/edge-dist/edgekd/models /content/edge-dist/edgekd/kd /content/edge-dist/edgekd/nas /content/edge-dist/scripts /content/edge-dist/logs /content/edge-dist/data

In [ ]:
%%writefile /content/edge-dist/edgekd/__init__.py
"""edgekd: knowledge distillation + hardware-aware NAS for edge devices."""

In [ ]:
%%writefile /content/edge-dist/edgekd/config.py
"""Training budgets.

QUICK is sized for a 4-core laptop CPU (whole pipeline ~1h). FULL is a
conventional schedule for a GPU machine. Select with `--profile full` on any script.
"""

PROFILES = {
    "quick": {
        "threads": 4,
        "batch_size": 128,
        # teacher: ImageNet-pretrained ResNet-50 fine-tuned on CIFAR-100
        "teacher": {"epochs": 2, "lr": 0.03, "train_subset": 25000, "input_res": 64, "freeze": True},
        # students (baseline and KD use identical budgets for a fair comparison)
        "student": {"epochs": 4, "lr": 0.1, "train_subset": 25000},
        # KD hyper-parameters
        "kd": {"temperature": 4.0, "alpha": 0.7, "beta": 1.0},
        # NAS (CIFAR-10)
        "nas": {"population": 8, "generations": 4, "proxy_epochs": 2, "proxy_train": 6000, "val": 2000,
                "lr": 0.05, "final_epochs": 5, "final_train_subset": None},
    },
    # Google Colab T4 GPU: whole pipeline ~30-35 min. Full CIFAR training sets, fine-tuned teacher at 96 px.
    "colab": {
        "threads": None,
        "batch_size": 256,
        "teacher": {"epochs": 3, "lr": 0.05, "train_subset": None, "input_res": 96, "freeze": False},
        "student": {"epochs": 10, "lr": 0.15, "train_subset": None},
        "kd": {"temperature": 4.0, "alpha": 0.7, "beta": 1.0},
        "nas": {"population": 8, "generations": 5, "proxy_epochs": 3, "proxy_train": 10000, "val": 5000,
                "lr": 0.1, "final_epochs": 15, "final_train_subset": None},
    },
    # few-minute end-to-end check that every stage runs (numbers are meaningless)
    "smoke": {
        "threads": None,
        "batch_size": 128,
        "teacher": {"epochs": 1, "lr": 0.03, "train_subset": 512, "input_res": 64, "freeze": True},
        "student": {"epochs": 1, "lr": 0.1, "train_subset": 512},
        "kd": {"temperature": 4.0, "alpha": 0.7, "beta": 1.0},
        "nas": {"population": 2, "generations": 1, "proxy_epochs": 1, "proxy_train": 512, "val": 256,
                "lr": 0.05, "final_epochs": 1, "final_train_subset": 1000},
    },
    "full": {
        "threads": None,
        "batch_size": 128,
        "teacher": {"epochs": 30, "lr": 0.05, "train_subset": None, "input_res": 224, "freeze": False},
        "student": {"epochs": 200, "lr": 0.1, "train_subset": None},
        "kd": {"temperature": 4.0, "alpha": 0.9, "beta": 1.0},
        "nas": {"population": 20, "generations": 15, "proxy_epochs": 5, "proxy_train": 20000, "val": 5000,
                "lr": 0.05, "final_epochs": 100, "final_train_subset": None},
    },
}

# Deployment constraints for hardware-aware NAS (batch-1 CPU inference, ONNX Runtime)
NAS_CONSTRAINTS = {"max_latency_ms": 3.0, "max_size_mb": 2.0, "max_peak_mem_mb": 25.0}


def get_profile(name):
    return PROFILES[name]

In [ ]:
%%writefile /content/edge-dist/edgekd/data.py
"""Fast in-memory CIFAR loaders.

The whole dataset is held as a uint8 tensor and augmentation (random crop with
4px padding + horizontal flip) is applied batch-wise on tensors. This avoids the
per-image PIL overhead of torchvision transforms, which dominates on CPU.
"""
import hashlib
import os
import pickle
import tarfile
import time
import urllib.request

import numpy as np
import torch
import torch.nn.functional as F

DATA_ROOT = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "data")

STATS = {
    "cifar100": ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
    "cifar10": ((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
}
NUM_CLASSES = {"cifar100": 100, "cifar10": 10}


ARCHIVES = {  # name -> (url, md5, extracted dir)
    "cifar100": ("https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz",
                 "eb9058c3a382ffc7106e4002c42a8d85", "cifar-100-python"),
    "cifar10": ("https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz",
                "c58f30108f718f92721af3b95e74349a", "cifar-10-batches-py"),
}
_REQUIRED = {
    "cifar100": ["train", "test"],
    "cifar10": [f"data_batch_{i}" for i in range(1, 6)] + ["test_batch"],
}


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _download(url, dst, timeout=30):
    """Stream url -> dst with a read timeout, progress output and HTTP Range resume of a partial file."""
    have = os.path.getsize(dst) if os.path.exists(dst) else 0
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    if have:
        req.add_header("Range", f"bytes={have}-")
    with urllib.request.urlopen(req, timeout=timeout) as r:  # timeout also applies to every read()
        if have and r.status != 206:  # server ignored Range: start over
            have = 0
        total = have + int(r.headers.get("Content-Length", 0))
        done, last, t0 = have, 0.0, time.time()
        with open(dst, "ab" if have else "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                done += len(chunk)
                if time.time() - last > 5:
                    last = time.time()
                    rate = (done - have) / max(last - t0, 1e-6) / 1e6
                    pct = f"{100 * done / total:5.1f}%" if total else ""
                    print(f"[data]   {done / 1e6:7.1f} / {total / 1e6:.1f} MB {pct}  {rate:.2f} MB/s", flush=True)
    if total and done < total:
        raise IOError(f"connection closed early ({done}/{total} bytes)")


def ensure_data(name, root=DATA_ROOT, retries=6):
    """Download + verify + extract a CIFAR archive if its files are missing. Raises loudly on failure."""
    url, md5, sub = ARCHIVES[name]
    base = os.path.join(root, sub)
    if all(os.path.exists(os.path.join(base, f)) for f in _REQUIRED[name]):
        return base
    os.makedirs(root, exist_ok=True)
    tgz = os.path.join(root, os.path.basename(url))
    for attempt in range(1, retries + 1):
        if os.path.exists(tgz) and _md5(tgz) == md5:
            break
        print(f"[data] downloading {name} (attempt {attempt}/{retries}) ...", flush=True)
        try:
            _download(url, tgz + ".part")  # keeps the .part on failure so the next attempt resumes
            os.replace(tgz + ".part", tgz)
        except Exception as e:  # network hiccup / stall: retry (resuming)
            print(f"[data] download failed: {e!r}", flush=True)
            time.sleep(5 * attempt)
        else:
            if _md5(tgz) != md5:  # corrupt: drop it so the next attempt starts clean
                print("[data] md5 mismatch, re-downloading", flush=True)
                os.remove(tgz)
    if not (os.path.exists(tgz) and _md5(tgz) == md5):
        raise RuntimeError(f"could not download a valid {name} archive from {url}")
    with tarfile.open(tgz, "r:gz") as t:
        t.extractall(root)
    missing = [f for f in _REQUIRED[name] if not os.path.exists(os.path.join(base, f))]
    if missing:
        raise RuntimeError(f"{name} extracted but files are missing: {missing}")
    print(f"[data] {name} ready at {base}", flush=True)
    return base


def _unpickle(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")


def load_raw(name, train, root=DATA_ROOT):
    """Return (images uint8 [N,3,32,32], labels int64 [N])."""
    ensure_data(name, root)
    if name == "cifar100":
        d = _unpickle(os.path.join(root, "cifar-100-python", "train" if train else "test"))
        x, y = d["data"], d["fine_labels"]
    elif name == "cifar10":
        base = os.path.join(root, "cifar-10-batches-py")
        files = [f"data_batch_{i}" for i in range(1, 6)] if train else ["test_batch"]
        xs, ys = [], []
        for f in files:
            d = _unpickle(os.path.join(base, f))
            xs.append(d["data"]); ys += d["labels"]
        x, y = np.concatenate(xs), ys
    else:
        raise ValueError(name)
    x = torch.from_numpy(np.asarray(x, dtype=np.uint8).reshape(-1, 3, 32, 32))
    return x, torch.tensor(y, dtype=torch.long)


def normalize(x_uint8, name):
    mean, std = STATS[name]
    x = x_uint8.float().div_(255.0)
    mean = torch.tensor(mean, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(std, device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std


def augment(x):
    """Random crop (pad 4) + horizontal flip on a float batch [B,3,H,W] (runs on x's device)."""
    b, _, h, w = x.shape
    d = x.device
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (b,), device=d)
    j = torch.randint(0, 9, (b,), device=d)
    # gather crops with unfold-free indexing
    rows = (i.view(b, 1) + torch.arange(h, device=d).view(1, h))  # [B,H]
    cols = (j.view(b, 1) + torch.arange(w, device=d).view(1, w))  # [B,W]
    xp = xp[torch.arange(b, device=d).view(b, 1, 1, 1), torch.arange(3, device=d).view(1, 3, 1, 1),
            rows.view(b, 1, h, 1), cols.view(b, 1, 1, w)]
    flip = torch.rand(b, device=d) < 0.5
    xp[flip] = xp[flip].flip(3)
    return xp


class TensorLoader:
    """Minimal iterable loader over in-memory tensors."""

    def __init__(self, x, y, name, batch_size=128, shuffle=False, train_aug=False, resolution=32, device=None):
        if device is not None:  # keep the whole (small) dataset resident on the GPU
            x, y = x.to(device), y.to(device)
        self.x, self.y, self.name = x, y, name
        self.bs, self.shuffle, self.aug, self.res = batch_size, shuffle, train_aug, resolution

    def __len__(self):
        return (len(self.x) + self.bs - 1) // self.bs

    def __iter__(self):
        n = len(self.x)
        d = self.x.device
        idx = torch.randperm(n, device=d) if self.shuffle else torch.arange(n, device=d)
        for s in range(0, n, self.bs):
            b = idx[s:s + self.bs]
            xb = normalize(self.x[b], self.name)
            if self.aug:
                xb = augment(xb)
            if self.res != 32:
                xb = F.interpolate(xb, size=(self.res, self.res), mode="bilinear", align_corners=False)
            yield xb, self.y[b]


def get_loaders(name="cifar100", batch_size=128, train_subset=None, test_subset=None,
                val_split=0, resolution=32, seed=0, device="auto"):
    """Build train / (val) / test loaders.

    train_subset: use only this many (class-balanced-ish random) training images.
    val_split:    hold out this many training images as a validation set (used by NAS).
    device:       "auto" -> CUDA if available (data resident on GPU), else CPU.
    """
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else None
    g = torch.Generator().manual_seed(seed)
    xtr, ytr = load_raw(name, True)
    xte, yte = load_raw(name, False)
    perm = torch.randperm(len(xtr), generator=g)
    xtr, ytr = xtr[perm], ytr[perm]
    val = None
    if val_split:
        val = TensorLoader(xtr[:val_split], ytr[:val_split], name, 512, resolution=resolution, device=device)
        xtr, ytr = xtr[val_split:], ytr[val_split:]
    if train_subset:
        xtr, ytr = xtr[:train_subset], ytr[:train_subset]
    if test_subset:
        p = torch.randperm(len(xte), generator=g)[:test_subset]
        xte, yte = xte[p], yte[p]
    train = TensorLoader(xtr, ytr, name, batch_size, shuffle=True, train_aug=True, resolution=resolution,
                         device=device)
    test = TensorLoader(xte, yte, name, 512, resolution=resolution, device=device)
    return (train, val, test) if val_split else (train, test)

In [ ]:
%%writefile /content/edge-dist/edgekd/utils.py
import json
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
CKPT_DIR = os.path.join(ROOT, "checkpoints")
RESULTS_DIR = os.path.join(ROOT, "results")
ONNX_DIR = os.path.join(ROOT, "onnx_models")
for _d in (CKPT_DIR, RESULTS_DIR, ONNX_DIR):
    os.makedirs(_d, exist_ok=True)

# Training runs on CUDA when available (mixed precision + channels_last), else CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if USE_AMP:
    torch.backends.cudnn.benchmark = True


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)


def grad_scaler():
    return torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)


def to_device(model):
    """Move a model to DEVICE (channels_last on GPU for faster convs)."""
    return model.to(DEVICE, memory_format=torch.channels_last) if USE_AMP else model.to(DEVICE)


def to_input(x):
    x = x.to(DEVICE, non_blocking=True)
    return x.contiguous(memory_format=torch.channels_last) if USE_AMP else x


def seed_all(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def set_threads(n=None):
    torch.set_num_threads(n or os.cpu_count())


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def logits_of(out):
    """Models may return (logits, features); return logits only."""
    return out[0] if isinstance(out, (tuple, list)) else out


@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval()
    dev = next(model.parameters()).device
    correct = total = 0
    for i, (x, y) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y = x.to(dev), y.to(dev)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            pred = logits_of(model(x)).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return 100.0 * correct / max(total, 1)


def make_optimizer(model, lr, epochs, steps_per_epoch, wd=5e-4, params=None):
    params = params if params is not None else [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd, nesterov=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(1, epochs * steps_per_epoch),
                                                pct_start=0.15)
    return opt, sched


def train_supervised(model, train_loader, test_loader, epochs, lr, log_name, label_smoothing=0.0,
                     params=None, eval_batches=None):
    """Plain cross-entropy training (teacher, baseline students, NAS candidates)."""
    to_device(model)
    crit = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    opt, sched = make_optimizer(model, lr, epochs, len(train_loader), params=params)
    scaler = grad_scaler()
    history = []
    for ep in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, y in train_loader:
            x, y = to_input(x), y.to(DEVICE)
            with autocast():
                loss = crit(logits_of(model(x)).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * y.numel(); n += y.numel()
        acc = evaluate(model, test_loader, eval_batches) if test_loader is not None else float("nan")
        rec = {"epoch": ep + 1, "train_loss": tot / n, "test_acc": acc, "epoch_time_s": time.time() - t0}
        history.append(rec)
        if log_name:
            print(f"[{log_name}] epoch {ep + 1}/{epochs} loss={rec['train_loss']:.4f} "
                  f"acc={acc:.2f}% ({rec['epoch_time_s']:.0f}s)", flush=True)
    return history


def save_checkpoint(model, path, **meta):
    torch.save({"state_dict": model.state_dict(), **meta}, path)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_json(path):
    with open(path) as f:
        return json.load(f)

In [ ]:
%%writefile /content/edge-dist/edgekd/inference.py
"""Unified inference pipeline for every trained model (KD students, baseline
students, NAS model, teacher) with two interchangeable back-ends:

    backend="torch" - PyTorch checkpoint (checkpoints/*.pt)
    backend="onnx"  - ONNX Runtime CPU session (onnx_models/*.onnx)

    p = Predictor("mobilenetv2_kd", backend="onnx")
    p.predict("cat.png", topk=3)          # file path / PIL image / HxWx3 uint8 array
    p.predict_batch(uint8_tensor_Nx3x32x32)
"""
import os
import pickle

import numpy as np
import torch
import torch.nn.functional as F

from .data import DATA_ROOT, STATS
from .utils import CKPT_DIR, ONNX_DIR

STUDENT_NAMES = ["mobilenetv2", "squeezenet", "efficientnet_lite0"]
MODEL_REGISTRY = {f"{s}_{m}": {"dataset": "cifar100", "ckpt": f"{s}_{m}.pt", "arch": s}
                  for s in STUDENT_NAMES for m in ("kd", "baseline")}
MODEL_REGISTRY["nas_best"] = {"dataset": "cifar10", "ckpt": "nas_best.pt", "arch": "nas"}
MODEL_REGISTRY["teacher_resnet50"] = {"dataset": "cifar100", "ckpt": "teacher_resnet50.pt", "arch": "teacher"}


def class_names(dataset):
    if dataset == "cifar100":
        with open(os.path.join(DATA_ROOT, "cifar-100-python", "meta"), "rb") as f:
            return pickle.load(f, encoding="latin1")["fine_label_names"]
    with open(os.path.join(DATA_ROOT, "cifar-10-batches-py", "batches.meta"), "rb") as f:
        return pickle.load(f, encoding="latin1")["label_names"]


def load_torch_model(name):
    """Rebuild the nn.Module for a registry entry and load its checkpoint."""
    from .models import ResNet50Teacher, build_student
    from .nas import build_model
    info = MODEL_REGISTRY[name]
    ck = torch.load(os.path.join(CKPT_DIR, info["ckpt"]), map_location="cpu", weights_only=False)
    if info["arch"] == "nas":
        model, res = build_model(ck["arch"], 10), ck["arch"]["resolution"]
    elif info["arch"] == "teacher":
        model, res = ResNet50Teacher(100, input_res=ck["input_res"], pretrained=False), 32
    else:
        model, res = build_student(info["arch"], 100), 32
    model.load_state_dict(ck["state_dict"])
    return model.eval(), res


class Predictor:
    def __init__(self, name, backend="onnx", threads=None):
        if name not in MODEL_REGISTRY:
            raise KeyError(f"unknown model {name}; choose from {sorted(MODEL_REGISTRY)}")
        self.name, self.backend = name, backend
        self.dataset = MODEL_REGISTRY[name]["dataset"]
        self.classes = class_names(self.dataset)
        mean, std = STATS[self.dataset]
        self.mean = np.array(mean, np.float32).reshape(1, 3, 1, 1)
        self.std = np.array(std, np.float32).reshape(1, 3, 1, 1)
        if backend == "torch":
            if threads:
                torch.set_num_threads(threads)
            self.model, self.resolution = load_torch_model(name)
        elif backend == "onnx":
            import onnxruntime as ort
            so = ort.SessionOptions()
            if threads:
                so.intra_op_num_threads = threads
            self.sess = ort.InferenceSession(os.path.join(ONNX_DIR, f"{name}.onnx"), so,
                                             providers=["CPUExecutionProvider"])
            shape = self.sess.get_inputs()[0].shape
            self.resolution = int(shape[-1])
            self.input_name = self.sess.get_inputs()[0].name
        else:
            raise ValueError(backend)

    # -------------------------------------------------------------- preprocessing
    def preprocess(self, images):
        """uint8 [N,3,H,W] (tensor or array) -> normalised float32 [N,3,R,R]."""
        x = torch.as_tensor(np.asarray(images)).float().div(255.0)
        if x.shape[-1] != self.resolution or x.shape[-2] != self.resolution:
            x = F.interpolate(x, size=(self.resolution, self.resolution), mode="bilinear", align_corners=False)
        return ((x.numpy() - self.mean) / self.std).astype(np.float32)

    @staticmethod
    def load_image(img):
        """File path / PIL image / HxWx3 array -> uint8 [1,3,32,32]."""
        from PIL import Image
        if isinstance(img, str):
            img = Image.open(img)
        if isinstance(img, Image.Image):
            img = np.asarray(img.convert("RGB").resize((32, 32), Image.BILINEAR))
        arr = np.asarray(img, dtype=np.uint8)
        return arr.transpose(2, 0, 1)[None]

    # -------------------------------------------------------------- inference
    def logits(self, x):
        if self.backend == "torch":
            with torch.no_grad():
                return self.model(torch.from_numpy(x)).numpy()
        return self.sess.run(None, {self.input_name: x})[0]

    def predict_batch(self, images_uint8):
        return self.logits(self.preprocess(images_uint8)).argmax(1)

    def predict(self, img, topk=5):
        z = self.logits(self.preprocess(self.load_image(img)))[0]
        p = np.exp(z - z.max()); p /= p.sum()
        idx = p.argsort()[::-1][:topk]
        return [(self.classes[i], float(p[i])) for i in idx]

In [ ]:
%%writefile /content/edge-dist/edgekd/models/__init__.py
from .students import STUDENTS, build_student  # noqa: F401
from .teacher import ResNet50Teacher  # noqa: F401

In [ ]:
%%writefile /content/edge-dist/edgekd/models/teacher.py
"""ResNet-50 teacher for CIFAR-100.

Initialised from ImageNet weights (transfer learning keeps CPU training short) and
fine-tuned on CIFAR-100. Inputs are 32x32 CIFAR tensors; they are bilinearly
upsampled to `input_res` inside the model so teacher and students share one
data pipeline. `forward(x, return_features=True)` also returns the layer4 feature
map used for feature-based distillation.
"""
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights


class ResNet50Teacher(nn.Module):
    feature_channels = 2048

    def __init__(self, num_classes=100, input_res=64, pretrained=True):
        super().__init__()
        self.input_res = input_res
        self.freeze_prefix = False
        net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        net.fc = nn.Linear(net.fc.in_features, num_classes)
        self.net = net

    def frozen_prefix(self):
        """Modules kept frozen during fast fine-tuning."""
        n = self.net
        return [n.conv1, n.bn1, n.layer1, n.layer2]

    def freeze(self):
        """Freeze stem..layer2 (weights and BN statistics) to cut CPU training cost."""
        self.freeze_prefix = True
        for m in self.frozen_prefix():
            for p in m.parameters():
                p.requires_grad = False
        return self

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_prefix:
            for m in self.frozen_prefix():
                m.eval()
        return self

    def forward(self, x, return_features=False):
        if x.shape[-1] != self.input_res:
            x = F.interpolate(x, size=(self.input_res, self.input_res), mode="bilinear", align_corners=False)
        n = self.net
        x = n.maxpool(n.relu(n.bn1(n.conv1(x))))
        x = n.layer4(n.layer3(n.layer2(n.layer1(x))))
        logits = n.fc(n.avgpool(x).flatten(1))
        return (logits, x) if return_features else logits

In [ ]:
%%writefile /content/edge-dist/edgekd/models/students.py
"""Lightweight student architectures, implemented from scratch and adapted to
32x32 CIFAR inputs (the ImageNet stem stride-2 is replaced by stride-1).

All students implement `forward(x, return_features=False)`; with
`return_features=True` they return `(logits, last_feature_map)` for
feature-based distillation. `feature_channels` gives the channel count of that map.
"""
import torch.nn as nn


def _make_divisible(v, divisor=8):
    new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
    return new_v + divisor if new_v < 0.9 * v else new_v


class ConvBNAct(nn.Sequential):
    def __init__(self, cin, cout, k=3, s=1, groups=1, act=True):
        layers = [nn.Conv2d(cin, cout, k, s, k // 2, groups=groups, bias=False), nn.BatchNorm2d(cout)]
        if act:
            layers.append(nn.ReLU6(inplace=True))
        super().__init__(*layers)


class InvertedResidual(nn.Module):
    """MobileNetV2 / EfficientNet-Lite MBConv block (no squeeze-excite)."""

    def __init__(self, cin, cout, stride, expand, k=3):
        super().__init__()
        hidden = cin * expand
        self.use_res = stride == 1 and cin == cout
        layers = []
        if expand != 1:
            layers.append(ConvBNAct(cin, hidden, 1))
        layers += [ConvBNAct(hidden, hidden, k, stride, groups=hidden), ConvBNAct(hidden, cout, 1, act=False)]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.block(x) if self.use_res else self.block(x)


class _Classifier(nn.Module):
    def _head(self, x, return_features):
        logits = self.classifier(self.pool(x).flatten(1))
        return (logits, x) if return_features else logits


# --------------------------------------------------------------------------- MobileNetV2
class MobileNetV2(_Classifier):
    # t, c, n, s
    CFG = [(1, 16, 1, 1), (6, 24, 2, 2), (6, 32, 3, 2), (6, 64, 4, 2),
           (6, 96, 3, 1), (6, 160, 3, 2), (6, 320, 1, 1)]

    def __init__(self, num_classes=100, width=1.0, last_channels=1280, dropout=0.2):
        super().__init__()
        cin = _make_divisible(32 * width)
        layers = [ConvBNAct(3, cin, 3, 1)]  # CIFAR: stride 1 stem
        for t, c, n, s in self.CFG:
            cout = _make_divisible(c * width)
            for i in range(n):
                layers.append(InvertedResidual(cin, cout, s if i == 0 else 1, t))
                cin = cout
        self.feature_channels = _make_divisible(last_channels * max(1.0, width))
        layers.append(ConvBNAct(cin, self.feature_channels, 1))
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.feature_channels, num_classes))
        _init(self)

    def forward(self, x, return_features=False):
        return self._head(self.features(x), return_features)


# --------------------------------------------------------------------------- SqueezeNet 1.1
class Fire(nn.Module):
    def __init__(self, cin, squeeze, e1, e3):
        super().__init__()
        self.squeeze = nn.Sequential(nn.Conv2d(cin, squeeze, 1), nn.BatchNorm2d(squeeze), nn.ReLU(inplace=True))
        self.e1 = nn.Sequential(nn.Conv2d(squeeze, e1, 1), nn.BatchNorm2d(e1), nn.ReLU(inplace=True))
        self.e3 = nn.Sequential(nn.Conv2d(squeeze, e3, 3, padding=1), nn.BatchNorm2d(e3), nn.ReLU(inplace=True))

    def forward(self, x):
        import torch
        s = self.squeeze(x)
        return torch.cat([self.e1(s), self.e3(s)], 1)


class SqueezeNet(_Classifier):
    """SqueezeNet v1.1 (with BatchNorm for stable short training)."""
    feature_channels = 512

    def __init__(self, num_classes=100, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),   # 32x32
            nn.MaxPool2d(3, 2, 1),                                                     # 16x16
            Fire(64, 16, 64, 64), Fire(128, 16, 64, 64),
            nn.MaxPool2d(3, 2, 1),                                                     # 8x8
            Fire(128, 32, 128, 128), Fire(256, 32, 128, 128),
            nn.MaxPool2d(3, 2, 1),                                                     # 4x4
            Fire(256, 48, 192, 192), Fire(384, 48, 192, 192),
            Fire(384, 64, 256, 256), Fire(512, 64, 256, 256),
        )
        # SqueezeNet classifies with a 1x1 conv followed by global average pooling
        self.classifier_conv = nn.Sequential(nn.Dropout(dropout), nn.Conv2d(512, num_classes, 1))
        self.pool = nn.AdaptiveAvgPool2d(1)
        _init(self)

    def forward(self, x, return_features=False):
        f = self.features(x)
        logits = self.pool(self.classifier_conv(f)).flatten(1)
        return (logits, f) if return_features else logits


# --------------------------------------------------------------------------- EfficientNet-Lite0
class EfficientNetLite0(_Classifier):
    """EfficientNet-Lite0: EfficientNet-B0 without squeeze-excite, ReLU6
    activations and a fixed (non-scaled) stem/head, as used for edge TPUs/mobile."""
    # expand, kernel, stride, out_channels, repeats
    CFG = [(1, 3, 1, 16, 1), (6, 3, 2, 24, 2), (6, 5, 2, 40, 2), (6, 3, 2, 80, 3),
           (6, 5, 1, 112, 3), (6, 5, 2, 192, 4), (6, 3, 1, 320, 1)]
    feature_channels = 1280

    def __init__(self, num_classes=100, dropout=0.2):
        super().__init__()
        layers = [ConvBNAct(3, 32, 3, 1)]  # CIFAR: stride 1 stem
        cin = 32
        for e, k, s, c, n in self.CFG:
            for i in range(n):
                layers.append(InvertedResidual(cin, c, s if i == 0 else 1, e, k))
                cin = c
        layers.append(ConvBNAct(cin, self.feature_channels, 1))
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.feature_channels, num_classes))
        _init(self)

    def forward(self, x, return_features=False):
        return self._head(self.features(x), return_features)


def _init(m):
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            nn.init.kaiming_normal_(mod.weight, mode="fan_out")
            if mod.bias is not None:
                nn.init.zeros_(mod.bias)
        elif isinstance(mod, nn.BatchNorm2d):
            nn.init.ones_(mod.weight); nn.init.zeros_(mod.bias)
        elif isinstance(mod, nn.Linear):
            nn.init.normal_(mod.weight, 0, 0.01); nn.init.zeros_(mod.bias)


STUDENTS = {
    "mobilenetv2": MobileNetV2,
    "squeezenet": SqueezeNet,
    "efficientnet_lite0": EfficientNetLite0,
}


def build_student(name, num_classes=100):
    return STUDENTS[name](num_classes=num_classes)

In [ ]:
%%writefile /content/edge-dist/edgekd/kd/__init__.py
from .losses import (DistillationLoss, FeatureDistiller, soft_target_loss,  # noqa: F401
                     softmax_with_temperature)
from .trainer import train_kd  # noqa: F401

In [ ]:
%%writefile /content/edge-dist/edgekd/kd/losses.py
"""Knowledge-distillation losses.

* soft-target KD with temperature-scaled softmax (Hinton et al., 2015)
* combined hard-label (CE) + soft-label (KL) objective
* feature-based distillation (FitNets-style hint loss) on intermediate feature maps
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def softmax_with_temperature(logits, T):
    """Temperature-scaled softmax: p_i = exp(z_i / T) / sum_j exp(z_j / T)."""
    return F.softmax(logits / T, dim=1)


def soft_target_loss(student_logits, teacher_logits, T):
    """KL(p_teacher^T || p_student^T) * T^2.

    The T^2 factor keeps gradient magnitudes comparable to the hard-label loss
    when the temperature changes.
    """
    log_p_s = F.log_softmax(student_logits / T, dim=1)
    p_t = softmax_with_temperature(teacher_logits, T)
    return F.kl_div(log_p_s, p_t, reduction="batchmean") * (T * T)


class FeatureDistiller(nn.Module):
    """Aligns a student feature map to a teacher feature map.

    A learnable 1x1 conv + BN regressor maps student channels -> teacher channels;
    both maps are pooled to a common spatial size and the MSE between
    channel-normalised features is minimised.
    """

    def __init__(self, student_channels, teacher_channels, spatial=2):
        super().__init__()
        self.regressor = nn.Sequential(nn.Conv2d(student_channels, teacher_channels, 1, bias=False),
                                       nn.BatchNorm2d(teacher_channels))
        self.spatial = spatial

    def forward(self, f_s, f_t):
        f_s = F.adaptive_avg_pool2d(self.regressor(f_s), self.spatial)
        f_t = F.adaptive_avg_pool2d(f_t, self.spatial)
        f_s = F.normalize(f_s.flatten(1), dim=1)
        f_t = F.normalize(f_t.flatten(1), dim=1)
        return (f_s - f_t).pow(2).sum(1).mean()


class DistillationLoss(nn.Module):
    """L = (1 - alpha) * CE(y, s) + alpha * T^2 * KL(t^T || s^T) + beta * L_feat.

    alpha = 0 -> plain hard-label training; beta = 0 -> logit-only KD.
    """

    def __init__(self, temperature=4.0, alpha=0.7, beta=1.0, feature_distiller=None):
        super().__init__()
        self.T, self.alpha, self.beta = temperature, alpha, beta
        self.feat = feature_distiller

    def forward(self, s_logits, t_logits, labels, s_feat=None, t_feat=None):
        hard = F.cross_entropy(s_logits, labels)
        soft = soft_target_loss(s_logits, t_logits, self.T)
        loss = (1 - self.alpha) * hard + self.alpha * soft
        parts = {"hard": hard.item(), "soft": soft.item()}
        if self.feat is not None and self.beta > 0 and s_feat is not None:
            fl = self.feat(s_feat, t_feat)
            loss = loss + self.beta * fl
            parts["feat"] = fl.item()
        return loss, parts

In [ ]:
%%writefile /content/edge-dist/edgekd/kd/trainer.py
import time

import torch

from ..utils import DEVICE, autocast, evaluate, grad_scaler, make_optimizer, to_device, to_input
from .losses import DistillationLoss, FeatureDistiller


def train_kd(student, teacher, train_loader, test_loader, epochs, lr, temperature=4.0, alpha=0.7,
             beta=1.0, log_name="kd"):
    """Distil a frozen teacher into a student with logit + feature distillation."""
    to_device(teacher).eval()
    to_device(student)
    for p in teacher.parameters():
        p.requires_grad = False
    distiller = FeatureDistiller(student.feature_channels, teacher.feature_channels) if beta > 0 else None
    crit = DistillationLoss(temperature, alpha, beta, distiller).to(DEVICE)
    params = list(student.parameters()) + (list(distiller.parameters()) if distiller else [])
    opt, sched = make_optimizer(student, lr, epochs, len(train_loader), params=params)
    scaler = grad_scaler()
    history = []
    for ep in range(epochs):
        student.train(); crit.train()
        t0, n, sums = time.time(), 0, {"loss": 0.0, "hard": 0.0, "soft": 0.0, "feat": 0.0}
        for x, y in train_loader:
            x, y = to_input(x), y.to(DEVICE)
            with autocast():
                with torch.no_grad():
                    t_logits, t_feat = teacher(x, return_features=True)
                s_logits, s_feat = student(x, return_features=True)
            # losses in fp32 for numerical stability of the temperature-scaled softmax
            loss, parts = crit(s_logits.float(), t_logits.float(), y, s_feat.float(), t_feat.float())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            b = y.numel(); n += b
            sums["loss"] += loss.item() * b
            for k, v in parts.items():
                sums[k] += v * b
        acc = evaluate(student, test_loader)
        rec = {"epoch": ep + 1, **{f"train_{k}": v / n for k, v in sums.items()}, "test_acc": acc,
               "epoch_time_s": time.time() - t0}
        history.append(rec)
        print(f"[{log_name}] epoch {ep + 1}/{epochs} loss={rec['train_loss']:.4f} "
              f"(hard={rec['train_hard']:.3f} soft={rec['train_soft']:.3f} feat={rec['train_feat']:.3f}) "
              f"acc={acc:.2f}% ({rec['epoch_time_s']:.0f}s)", flush=True)
    return history

In [ ]:
%%writefile /content/edge-dist/edgekd/nas/__init__.py
from .evolution import EvolutionarySearch  # noqa: F401
from .generator import build_model  # noqa: F401
from .hardware import check_constraints, export_onnx, profile_model, profile_onnx  # noqa: F401
from .search_space import SPACE, sample_architecture  # noqa: F401

In [ ]:
%%writefile /content/edge-dist/edgekd/nas/search_space.py
"""Searchable CNN architecture space.

An architecture is a JSON-serialisable dict:

    {
      "resolution": 32,                       # variable input resolution
      "stem_channels": 24,
      "stages": [                             # variable depth (#stages, #blocks per stage)
        {"blocks": 2, "channels": 32,          # variable filters
         "kernel": 3,                          # 3 / 5 / 7
         "block": "dwsep",                     # conv | dwsep (depthwise-separable) | mbconv
         "expand": 4,                          # only used by mbconv
         "act": "hardswish"},                  # relu | relu6 | hardswish | silu | leaky_relu
        ...
      ]
    }

Every stage starts with a stride-2 block, so the final spatial size is
resolution / 2**num_stages.
"""
import copy
import json
import random

SPACE = {
    "resolution": [24, 28, 32],
    "stem_channels": [16, 24, 32],
    "num_stages": [2, 3, 4],
    "blocks": [1, 2, 3],
    "channels": [16, 24, 32, 48, 64, 96, 128],
    "kernel": [3, 5, 7],
    "block": ["conv", "dwsep", "mbconv"],
    "expand": [2, 4, 6],
    "act": ["relu", "relu6", "hardswish", "silu", "leaky_relu"],
}


def _sample_stage(rng, prev_channels):
    # channels are non-decreasing across stages (standard CNN design prior)
    ch = rng.choice([c for c in SPACE["channels"] if c >= prev_channels] or [SPACE["channels"][-1]])
    return {"blocks": rng.choice(SPACE["blocks"]), "channels": ch, "kernel": rng.choice(SPACE["kernel"]),
            "block": rng.choice(SPACE["block"]), "expand": rng.choice(SPACE["expand"]),
            "act": rng.choice(SPACE["act"])}


def sample_architecture(rng=random):
    arch = {"resolution": rng.choice(SPACE["resolution"]), "stem_channels": rng.choice(SPACE["stem_channels"]),
            "stages": []}
    prev = arch["stem_channels"]
    for _ in range(rng.choice(SPACE["num_stages"])):
        st = _sample_stage(rng, prev)
        arch["stages"].append(st)
        prev = st["channels"]
    return arch


def _sort_channels(arch):
    chans = sorted(s["channels"] for s in arch["stages"])
    for s, c in zip(arch["stages"], chans):
        s["channels"] = c
    return arch


def mutate(arch, rng=random, n_mutations=None):
    """Randomly change 1-2 genes (a global gene, a stage gene, or add/remove a stage)."""
    a = copy.deepcopy(arch)
    for _ in range(n_mutations or rng.choice([1, 2])):
        r = rng.random()
        if r < 0.15:
            a["resolution"] = rng.choice(SPACE["resolution"])
        elif r < 0.25:
            a["stem_channels"] = rng.choice(SPACE["stem_channels"])
        elif r < 0.35:
            n = len(a["stages"])
            if n < max(SPACE["num_stages"]) and (rng.random() < 0.5 or n == min(SPACE["num_stages"])):
                a["stages"].append(_sample_stage(rng, a["stages"][-1]["channels"]))
            elif n > min(SPACE["num_stages"]):
                a["stages"].pop(rng.randrange(n))
        else:
            st = rng.choice(a["stages"])
            key = rng.choice(["blocks", "channels", "kernel", "block", "expand", "act"])
            st[key] = rng.choice(SPACE[key])
    return _sort_channels(a)


def crossover(a, b, rng=random):
    """Uniform crossover: global genes and aligned stages are inherited from either parent."""
    child = {"resolution": rng.choice([a, b])["resolution"],
             "stem_channels": rng.choice([a, b])["stem_channels"]}
    n = rng.choice([len(a["stages"]), len(b["stages"])])
    stages = []
    for i in range(n):
        opts = [p["stages"][i] for p in (a, b) if i < len(p["stages"])]
        stages.append(copy.deepcopy(rng.choice(opts)))
    child["stages"] = stages
    return _sort_channels(child)


def arch_key(arch):
    """Canonical string used to de-duplicate architectures."""
    a = copy.deepcopy(arch)
    for s in a["stages"]:
        if s["block"] != "mbconv":
            s["expand"] = None
    return json.dumps(a, sort_keys=True)

In [ ]:
%%writefile /content/edge-dist/edgekd/nas/generator.py
"""Automated architecture generator: turns an architecture config (see
search_space.py) into a trainable `nn.Module`."""
import torch.nn as nn
import torch.nn.functional as F

ACTIVATIONS = {
    "relu": lambda: nn.ReLU(inplace=True),
    "relu6": lambda: nn.ReLU6(inplace=True),
    "hardswish": lambda: nn.Hardswish(inplace=True),
    "silu": lambda: nn.SiLU(inplace=True),
    "leaky_relu": lambda: nn.LeakyReLU(0.1, inplace=True),
}


def conv_bn(cin, cout, k, s, act=None, groups=1):
    layers = [nn.Conv2d(cin, cout, k, s, k // 2, groups=groups, bias=False), nn.BatchNorm2d(cout)]
    if act:
        layers.append(ACTIVATIONS[act]())
    return nn.Sequential(*layers)


class ConvBlock(nn.Module):
    """Standard KxK convolution block with a residual connection when shapes match."""

    def __init__(self, cin, cout, k, s, act, **_):
        super().__init__()
        self.body = conv_bn(cin, cout, k, s, act)
        self.res = s == 1 and cin == cout

    def forward(self, x):
        return x + self.body(x) if self.res else self.body(x)


class DWSepBlock(nn.Module):
    """Depthwise-separable conv: KxK depthwise -> 1x1 pointwise."""

    def __init__(self, cin, cout, k, s, act, **_):
        super().__init__()
        self.body = nn.Sequential(conv_bn(cin, cin, k, s, act, groups=cin), conv_bn(cin, cout, 1, 1, act))
        self.res = s == 1 and cin == cout

    def forward(self, x):
        return x + self.body(x) if self.res else self.body(x)


class MBConvBlock(nn.Module):
    """Inverted residual: 1x1 expand -> KxK depthwise -> 1x1 linear projection."""

    def __init__(self, cin, cout, k, s, act, expand=4):
        super().__init__()
        h = cin * expand
        self.body = nn.Sequential(conv_bn(cin, h, 1, 1, act), conv_bn(h, h, k, s, act, groups=h),
                                  conv_bn(h, cout, 1, 1, None))
        self.res = s == 1 and cin == cout

    def forward(self, x):
        return x + self.body(x) if self.res else self.body(x)


BLOCKS = {"conv": ConvBlock, "dwsep": DWSepBlock, "mbconv": MBConvBlock}


class GeneratedNet(nn.Module):
    def __init__(self, arch, num_classes=10):
        super().__init__()
        self.arch = arch
        self.resolution = arch["resolution"]
        first_act = arch["stages"][0]["act"]
        layers = [conv_bn(3, arch["stem_channels"], 3, 1, first_act)]
        cin = arch["stem_channels"]
        for st in arch["stages"]:
            for i in range(st["blocks"]):
                layers.append(BLOCKS[st["block"]](cin, st["channels"], st["kernel"], 2 if i == 0 else 1,
                                                  st["act"], expand=st["expand"]))
                cin = st["channels"]
        self.features = nn.Sequential(*layers)
        self.feature_channels = cin
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")

    def forward(self, x, return_features=False):
        # inputs of a different size are resized to the searched resolution
        if x.shape[-1] != self.resolution:
            x = F.interpolate(x, size=(self.resolution, self.resolution), mode="bilinear", align_corners=False)
        f = self.features(x)
        logits = self.classifier(self.pool(f).flatten(1))
        return (logits, f) if return_features else logits


def build_model(arch, num_classes=10):
    return GeneratedNet(arch, num_classes)

In [ ]:
%%writefile /content/edge-dist/edgekd/nas/hardware.py
"""Hardware profiling for the hardware-aware search.

Each candidate is exported to ONNX and profiled on CPU with ONNX Runtime
(batch 1, single intra-op thread to mimic a small edge core):

* latency      - median wall-clock over N runs after warm-up
* storage size - size of the exported .onnx file
* peak memory  - peak RSS increase (session creation + inference) measured in a
                 fresh subprocess: the kernel's peak-RSS counter (VmHWM) is reset
                 via /proc/self/clear_refs after imports (Linux only; `None` where
                 unsupported)
"""
import copy
import json
import os
import subprocess
import sys
import tempfile

import torch

WORKER = os.path.abspath(__file__)


def export_onnx(model, path, resolution, opset=17):
    """Export an fp32 CPU copy of `model` (the original may live on the GPU)."""
    model = copy.deepcopy(model).cpu().float().eval()
    model = model.to(memory_format=torch.contiguous_format)
    dummy = torch.randn(1, 3, resolution, resolution)
    torch.onnx.export(model, dummy, path, input_names=["input"], output_names=["logits"],
                      dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=opset,
                      dynamo=False)
    return path


def profile_onnx(path, resolution, threads=1, warmup=10, runs=50):
    """Run the profiling worker in a clean subprocess and return its metrics."""
    out = subprocess.run([sys.executable, WORKER, path, str(resolution), str(threads), str(warmup), str(runs)],
                         capture_output=True, text=True, timeout=300)
    if out.returncode != 0:
        raise RuntimeError(out.stderr[-2000:])
    m = json.loads(out.stdout.strip().splitlines()[-1])
    m["size_mb"] = os.path.getsize(path) / 2 ** 20
    return m


def profile_model(model, resolution, keep_path=None, **kw):
    """Export `model` to ONNX (temp file unless keep_path) and profile it."""
    if keep_path:
        export_onnx(model, keep_path, resolution)
        return profile_onnx(keep_path, resolution, **kw)
    with tempfile.TemporaryDirectory() as d:
        p = export_onnx(model, os.path.join(d, "m.onnx"), resolution)
        return profile_onnx(p, resolution, **kw)


def check_constraints(metrics, constraints):
    """Return a list of violated constraints (empty -> deployable)."""
    v = []
    if metrics["latency_ms"] > constraints["max_latency_ms"]:
        v.append(f"latency {metrics['latency_ms']:.2f}ms > {constraints['max_latency_ms']}ms")
    if metrics["size_mb"] > constraints["max_size_mb"]:
        v.append(f"size {metrics['size_mb']:.2f}MB > {constraints['max_size_mb']}MB")
    pm = metrics.get("peak_mem_mb")
    if pm is not None and pm > constraints["max_peak_mem_mb"]:
        v.append(f"peak mem {pm:.1f}MB > {constraints['max_peak_mem_mb']}MB")
    return v


def _worker(path, resolution, threads, warmup, runs):
    import time

    import numpy as np
    import onnxruntime as ort

    def _status_kb(field):
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith(field):
                    return int(line.split()[1])

    try:  # Linux: reset the peak-RSS (VmHWM) counter so it reflects only the model
        with open("/proc/self/clear_refs", "w") as f:
            f.write("5")

        def rss_peak_mb():
            return _status_kb("VmHWM") / 1024
        base = _status_kb("VmRSS") / 1024
    except OSError:  # other OSes: peak memory unsupported
        rss_peak_mb, base = None, None

    x = np.random.randn(1, 3, resolution, resolution).astype(np.float32)
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    for _ in range(warmup):
        sess.run(None, {"input": x})
    times = []
    for _ in range(runs):
        t = time.perf_counter()
        sess.run(None, {"input": x})
        times.append((time.perf_counter() - t) * 1000)
    peak = (rss_peak_mb() - base) if rss_peak_mb else None
    print(json.dumps({"latency_ms": float(np.median(times)), "latency_p90_ms": float(np.percentile(times, 90)),
                      "peak_mem_mb": peak}))


if __name__ == "__main__":
    a = sys.argv[1:]
    _worker(a[0], int(a[1]), int(a[2]), int(a[3]), int(a[4]))

In [ ]:
%%writefile /content/edge-dist/edgekd/nas/evolution.py
"""Hardware-aware evolutionary architecture search ((mu + lambda) with
tournament selection, uniform crossover and mutation).

Each candidate is first profiled on CPU (latency / size / peak memory).
Candidates violating the deployment constraints are rejected *before* training,
so no compute is spent on undeployable networks. Deployable candidates are
trained briefly on a CIFAR-10 proxy split and scored by validation accuracy.
"""
import random
import time

from ..utils import count_params, evaluate, train_supervised
from .generator import build_model
from .hardware import check_constraints, profile_model
from .search_space import arch_key, crossover, mutate, sample_architecture


class EvolutionarySearch:
    def __init__(self, train_loader, val_loader, constraints, population=8, generations=4, proxy_epochs=1,
                 lr=0.05, seed=0, log=print):
        self.train, self.val = train_loader, val_loader
        self.constraints = constraints
        self.P, self.G, self.E, self.lr = population, generations, proxy_epochs, lr
        self.rng = random.Random(seed)
        self.cache = {}       # arch_key -> record
        self.history = []     # every evaluated candidate, in order
        self.log = log

    # ------------------------------------------------------------------ evaluation
    def evaluate(self, arch, generation):
        key = arch_key(arch)
        if key in self.cache:
            return self.cache[key]
        model = build_model(arch, 10)
        rec = {"id": len(self.history), "generation": generation, "arch": arch, "params": count_params(model)}
        hw = profile_model(model, arch["resolution"])
        rec.update(latency_ms=hw["latency_ms"], size_mb=hw["size_mb"], peak_mem_mb=hw["peak_mem_mb"])
        violations = check_constraints(hw, self.constraints)
        if violations:
            rec.update(status="rejected", violations="; ".join(violations), val_acc=None, fitness=float("-inf"),
                       train_time_s=0.0)
        else:
            t0 = time.time()
            train_supervised(model, self.train, None, self.E, self.lr, log_name=None)
            acc = evaluate(model, self.val)
            rec.update(status="accepted", violations="", val_acc=acc, fitness=acc, train_time_s=time.time() - t0)
        self.cache[key] = rec
        self.history.append(rec)
        self.log(f"  [gen {generation}] #{rec['id']:02d} {rec['status']:8s} "
                 f"params={rec['params'] / 1e3:6.1f}K lat={rec['latency_ms']:.2f}ms size={rec['size_mb']:.2f}MB "
                 f"mem={rec['peak_mem_mb'] or 0:.1f}MB "
                 + (f"val_acc={rec['val_acc']:.2f}%" if rec["val_acc"] is not None else rec["violations"]))
        return rec

    # ------------------------------------------------------------------ operators
    def _tournament(self, pop, k=3):
        return max(self.rng.sample(pop, min(k, len(pop))), key=lambda r: r["fitness"])

    def _offspring(self, pop):
        if self.rng.random() < 0.5 and len(pop) > 1:
            child = crossover(self._tournament(pop)["arch"], self._tournament(pop)["arch"], self.rng)
            child = mutate(child, self.rng, n_mutations=1)
        else:
            child = mutate(self._tournament(pop)["arch"], self.rng)
        return child

    # ------------------------------------------------------------------ main loop
    def run(self):
        self.log(f"generation 0: sampling {self.P} deployable architectures")
        pop, tries = [], 0
        while len(pop) < self.P and tries < self.P * 6:
            tries += 1
            rec = self.evaluate(sample_architecture(self.rng), 0)
            if rec["status"] == "accepted" and rec not in pop:
                pop.append(rec)
        for g in range(1, self.G + 1):
            self.log(f"generation {g}: best so far {max(r['fitness'] for r in pop):.2f}%")
            children, tries = [], 0
            while len(children) < self.P and tries < self.P * 6:
                tries += 1
                arch = self._offspring(pop)
                if arch_key(arch) in self.cache:
                    continue
                rec = self.evaluate(arch, g)
                if rec["status"] == "accepted":
                    children.append(rec)
            pop = sorted(pop + children, key=lambda r: r["fitness"], reverse=True)[: self.P]  # (mu + lambda)
        best = max(pop, key=lambda r: r["fitness"])
        return best, self.history

In [ ]:
%%writefile /content/edge-dist/scripts/train_teacher.py
"""Train (fine-tune) the ResNet-50 teacher on CIFAR-100.

    python scripts/train_teacher.py [--profile quick|full]
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from edgekd.config import get_profile  # noqa: E402
from edgekd.data import get_loaders  # noqa: E402
from edgekd.models import ResNet50Teacher  # noqa: E402
from edgekd.utils import (CKPT_DIR, RESULTS_DIR, count_params, save_checkpoint, save_json,  # noqa: E402
                          seed_all, set_threads, train_supervised)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", default="quick")
    ap.add_argument("--epochs", type=int)
    args = ap.parse_args()
    P = get_profile(args.profile); T = P["teacher"]
    seed_all(0); set_threads(P["threads"])

    train, test = get_loaders("cifar100", P["batch_size"], train_subset=T["train_subset"])
    model = ResNet50Teacher(100, input_res=T["input_res"], pretrained=True)
    if T["freeze"]:
        model.freeze()
    print(f"ResNet-50 teacher: {count_params(model) / 1e6:.2f}M params, input {T['input_res']}px, "
          f"{len(train.x)} train images", flush=True)
    hist = train_supervised(model, train, test, args.epochs or T["epochs"], T["lr"], "teacher",
                            label_smoothing=0.1)
    save_checkpoint(model, os.path.join(CKPT_DIR, "teacher_resnet50.pt"), input_res=T["input_res"],
                    test_acc=hist[-1]["test_acc"])
    save_json({"model": "resnet50", "params": count_params(model), "history": hist},
              os.path.join(RESULTS_DIR, "teacher_history.json"))
    print(f"teacher final test acc {hist[-1]['test_acc']:.2f}%")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/edge-dist/scripts/train_students.py
"""Train the three student networks on CIFAR-100, either as plain baselines
(hard labels only) or with knowledge distillation from the ResNet-50 teacher.

    python scripts/train_students.py --mode baseline             # all three, no KD
    python scripts/train_students.py --mode kd                   # all three, with KD
    python scripts/train_students.py --mode kd --models squeezenet --alpha 0.9 --beta 0
"""
import argparse
import os
import sys

import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from edgekd.config import get_profile  # noqa: E402
from edgekd.data import get_loaders  # noqa: E402
from edgekd.kd import train_kd  # noqa: E402
from edgekd.models import STUDENTS, ResNet50Teacher, build_student  # noqa: E402
from edgekd.utils import (CKPT_DIR, RESULTS_DIR, count_params, save_checkpoint, save_json,  # noqa: E402
                          seed_all, set_threads, train_supervised)


def load_teacher():
    ck = torch.load(os.path.join(CKPT_DIR, "teacher_resnet50.pt"), map_location="cpu")
    t = ResNet50Teacher(100, input_res=ck["input_res"], pretrained=False)
    t.load_state_dict(ck["state_dict"])
    return t.eval()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["baseline", "kd"], required=True)
    ap.add_argument("--models", nargs="+", default=list(STUDENTS))
    ap.add_argument("--profile", default="quick")
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--temperature", type=float)
    ap.add_argument("--alpha", type=float)
    ap.add_argument("--beta", type=float)
    args = ap.parse_args()
    P = get_profile(args.profile); S = P["student"]; K = dict(P["kd"])
    for k in ("temperature", "alpha", "beta"):
        if getattr(args, k) is not None:
            K[k] = getattr(args, k)
    epochs = args.epochs or S["epochs"]
    set_threads(P["threads"])
    teacher = load_teacher() if args.mode == "kd" else None

    for name in args.models:
        seed_all(0)  # same init + data order for baseline and KD runs
        train, test = get_loaders("cifar100", P["batch_size"], train_subset=S["train_subset"])
        model = build_student(name, 100)
        tag = f"{name}_{args.mode}"
        print(f"== {tag}: {count_params(model) / 1e6:.2f}M params, {epochs} epochs", flush=True)
        if args.mode == "baseline":
            hist = train_supervised(model, train, test, epochs, S["lr"], tag)
        else:
            hist = train_kd(model, teacher, train, test, epochs, S["lr"], log_name=tag, **K)
        save_checkpoint(model, os.path.join(CKPT_DIR, f"{tag}.pt"), arch=name, mode=args.mode,
                        test_acc=hist[-1]["test_acc"], kd=K if args.mode == "kd" else None)
        save_json({"model": name, "mode": args.mode, "params": count_params(model),
                   "kd": K if args.mode == "kd" else None, "history": hist},
                  os.path.join(RESULTS_DIR, f"{tag}_history.json"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/edge-dist/scripts/nas_search.py
"""Hardware-aware evolutionary NAS on CIFAR-10.

    python scripts/nas_search.py [--profile quick|full] [--max-latency-ms 3 --max-size-mb 2 --max-mem-mb 25]

Outputs:
    results/nas_history.csv / .json  - every candidate (config, latency, size, memory, status, accuracy)
    results/nas_best_arch.json       - best deployable architecture
"""
import argparse
import json
import os
import sys
import time

import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from edgekd.config import NAS_CONSTRAINTS, get_profile  # noqa: E402
from edgekd.data import get_loaders  # noqa: E402
from edgekd.nas import EvolutionarySearch  # noqa: E402
from edgekd.utils import RESULTS_DIR, save_json, seed_all, set_threads  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", default="quick")
    ap.add_argument("--max-latency-ms", type=float, default=NAS_CONSTRAINTS["max_latency_ms"])
    ap.add_argument("--max-size-mb", type=float, default=NAS_CONSTRAINTS["max_size_mb"])
    ap.add_argument("--max-mem-mb", type=float, default=NAS_CONSTRAINTS["max_peak_mem_mb"])
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    P = get_profile(args.profile); N = P["nas"]
    seed_all(args.seed); set_threads(P["threads"])
    constraints = {"max_latency_ms": args.max_latency_ms, "max_size_mb": args.max_size_mb,
                   "max_peak_mem_mb": args.max_mem_mb}
    print("constraints:", constraints, flush=True)

    train, val, _ = get_loaders("cifar10", P["batch_size"], train_subset=N["proxy_train"], val_split=N["val"],
                                test_subset=1, seed=args.seed)
    search = EvolutionarySearch(train, val, constraints, N["population"], N["generations"], N["proxy_epochs"],
                                N["lr"], seed=args.seed, log=lambda s: print(s, flush=True))
    t0 = time.time()
    best, history = search.run()
    elapsed = time.time() - t0

    rows = [{**{k: v for k, v in r.items() if k != "arch"}, "arch": json.dumps(r["arch"]),
             "resolution": r["arch"]["resolution"], "num_stages": len(r["arch"]["stages"]),
             "depth": sum(s["blocks"] for s in r["arch"]["stages"])} for r in history]
    pd.DataFrame(rows).to_csv(os.path.join(RESULTS_DIR, "nas_history.csv"), index=False)
    save_json({"constraints": constraints, "search_time_s": elapsed, "history": history},
              os.path.join(RESULTS_DIR, "nas_history.json"))
    save_json({"arch": best["arch"], "proxy_val_acc": best["val_acc"], "latency_ms": best["latency_ms"],
               "size_mb": best["size_mb"], "peak_mem_mb": best["peak_mem_mb"], "params": best["params"],
               "constraints": constraints}, os.path.join(RESULTS_DIR, "nas_best_arch.json"))
    n_rej = sum(r["status"] == "rejected" for r in history)
    print(f"\nsearch done in {elapsed / 60:.1f} min: {len(history)} candidates, {n_rej} rejected by constraints")
    print("best:", json.dumps(best["arch"]))
    print(f"proxy val acc {best['val_acc']:.2f}%  latency {best['latency_ms']:.2f}ms  size {best['size_mb']:.2f}MB")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/edge-dist/scripts/nas_train_best.py
"""Train the best discovered architecture from scratch on CIFAR-10.

    python scripts/nas_train_best.py [--profile quick|full] [--arch results/nas_best_arch.json]
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from edgekd.config import get_profile  # noqa: E402
from edgekd.data import get_loaders  # noqa: E402
from edgekd.nas import build_model  # noqa: E402
from edgekd.utils import (CKPT_DIR, RESULTS_DIR, count_params, load_json, save_checkpoint,  # noqa: E402
                          save_json, seed_all, set_threads, train_supervised)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", default="quick")
    ap.add_argument("--arch", default=os.path.join(RESULTS_DIR, "nas_best_arch.json"))
    ap.add_argument("--epochs", type=int)
    args = ap.parse_args()
    P = get_profile(args.profile); N = P["nas"]
    seed_all(0); set_threads(P["threads"])
    arch = load_json(args.arch)["arch"]
    model = build_model(arch, 10)  # fresh weights: trained from scratch
    train, test = get_loaders("cifar10", P["batch_size"], train_subset=N["final_train_subset"])
    epochs = args.epochs or N["final_epochs"]
    print(f"NAS best: {count_params(model) / 1e3:.1f}K params, res {arch['resolution']}, {epochs} epochs", flush=True)
    hist = train_supervised(model, train, test, epochs, N["lr"] * 2, "nas_best")
    save_checkpoint(model, os.path.join(CKPT_DIR, "nas_best.pt"), arch=arch, test_acc=hist[-1]["test_acc"])
    save_json({"arch": arch, "params": count_params(model), "history": hist},
              os.path.join(RESULTS_DIR, "nas_best_train_history.json"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/edge-dist/scripts/export_onnx.py
"""Export every trained student (KD + baseline) and the NAS model to ONNX and
verify ONNX Runtime outputs against PyTorch.

    python scripts/export_onnx.py [--models mobilenetv2_kd nas_best ...] [--include-teacher]
"""
import argparse
import os
import sys

import numpy as np
import onnx
import onnxruntime as ort
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from edgekd.inference import MODEL_REGISTRY, load_torch_model  # noqa: E402
from edgekd.nas.hardware import export_onnx  # noqa: E402
from edgekd.utils import CKPT_DIR, ONNX_DIR  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+")
    ap.add_argument("--include-teacher", action="store_true")
    args = ap.parse_args()
    names = args.models or [n for n in MODEL_REGISTRY if n != "teacher_resnet50" or args.include_teacher]
    for name in names:
        if not os.path.exists(os.path.join(CKPT_DIR, MODEL_REGISTRY[name]["ckpt"])):
            print(f"skip {name}: no checkpoint"); continue
        model, res = load_torch_model(name)
        path = os.path.join(ONNX_DIR, f"{name}.onnx")
        export_onnx(model, path, res)
        onnx.checker.check_model(onnx.load(path))
        x = torch.randn(4, 3, res, res)
        with torch.no_grad():
            ref = model(x).numpy()
        out = ort.InferenceSession(path, providers=["CPUExecutionProvider"]).run(None, {"input": x.numpy()})[0]
        diff = float(np.abs(ref - out).max())
        print(f"{name:28s} -> {path} ({os.path.getsize(path) / 2**20:.2f} MB), max |torch-ort| = {diff:.2e}")
        assert diff < 1e-3, "ONNX output mismatch"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/edge-dist/scripts/infer.py
"""Unified CPU inference CLI for all student models and the NAS model.

    # classify an image file with a distilled student via ONNX Runtime
    python scripts/infer.py --model mobilenetv2_kd --image path/to/img.png

    # evaluate test accuracy of one / all models (ONNX Runtime or PyTorch backend)
    python scripts/infer.py --model all --eval --backend onnx
    python scripts/infer.py --model nas_best --eval --backend torch

    # show predictions for a few CIFAR test images
    python scripts/infer.py --model squeezenet_kd --samples 5
"""
import argparse
import os
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from edgekd.data import load_raw  # noqa: E402
from edgekd.inference import MODEL_REGISTRY, Predictor  # noqa: E402
from edgekd.utils import ONNX_DIR  # noqa: E402


def available(backend):
    names = [n for n in MODEL_REGISTRY if n != "teacher_resnet50"]
    if backend == "onnx":
        names = [n for n in names if os.path.exists(os.path.join(ONNX_DIR, f"{n}.onnx"))]
    return names


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="all", help=f"one of {sorted(MODEL_REGISTRY)} or 'all'")
    ap.add_argument("--backend", choices=["onnx", "torch"], default="onnx")
    ap.add_argument("--image")
    ap.add_argument("--eval", action="store_true", help="accuracy on the test set")
    ap.add_argument("--samples", type=int, default=0, help="print predictions for N test images")
    ap.add_argument("--threads", type=int)
    args = ap.parse_args()
    names = available(args.backend) if args.model == "all" else [args.model]

    for name in names:
        p = Predictor(name, args.backend, args.threads)
        print(f"== {name} [{args.backend}, {p.dataset}, input {p.resolution}px]")
        if args.image:
            for cls, prob in p.predict(args.image, topk=5):
                print(f"   {cls:15s} {prob:.3f}")
        if args.samples or args.eval:
            x, y = load_raw(p.dataset, train=False)
            if args.samples:
                pred = p.predict_batch(x[:args.samples])
                for i in range(args.samples):
                    print(f"   test[{i}] true={p.classes[y[i]]:12s} pred={p.classes[pred[i]]}")
            if args.eval:
                t0, correct = time.time(), 0
                for s in range(0, len(x), 500):
                    correct += (p.predict_batch(x[s:s + 500]) == y[s:s + 500].numpy()).sum()
                dt = time.time() - t0
                print(f"   test accuracy {100 * correct / len(x):.2f}%  ({len(x) / dt:.0f} img/s)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/edge-dist/scripts/benchmark.py
"""Automated benchmark of all trained models -> results/benchmark.csv

Per model: parameters, ONNX size, test accuracy (ONNX Runtime), batch-1 CPU
latency for PyTorch and ONNX Runtime, and ONNX Runtime throughput (batch 64).

    python scripts/benchmark.py [--runs 50]
"""
import argparse
import os
import sys
import time

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from edgekd.data import load_raw  # noqa: E402
from edgekd.inference import MODEL_REGISTRY, Predictor  # noqa: E402
from edgekd.nas.hardware import profile_onnx  # noqa: E402
from edgekd.utils import CKPT_DIR, ONNX_DIR, RESULTS_DIR, count_params, load_json  # noqa: E402


def latency_ms(fn, runs, warmup=10):
    for _ in range(warmup):
        fn()
    t = []
    for _ in range(runs):
        s = time.perf_counter(); fn(); t.append((time.perf_counter() - s) * 1000)
    return float(np.median(t))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", type=int, default=50)
    ap.add_argument("--threads", type=int, default=1)
    args = ap.parse_args()
    torch.set_num_threads(args.threads)
    rows = []
    data = {d: load_raw(d, train=False) for d in ("cifar100", "cifar10")}
    teacher_acc = None
    tp = os.path.join(CKPT_DIR, "teacher_resnet50.pt")
    if os.path.exists(tp):
        teacher_acc = torch.load(tp, map_location="cpu", weights_only=False)["test_acc"]

    for name in MODEL_REGISTRY:
        if name == "teacher_resnet50" or not os.path.exists(os.path.join(ONNX_DIR, f"{name}.onnx")):
            continue
        ort_p = Predictor(name, "onnx", threads=args.threads)
        pt_p = Predictor(name, "torch", threads=args.threads)
        x, y = data[ort_p.dataset]
        correct = sum((ort_p.predict_batch(x[s:s + 500]) == y[s:s + 500].numpy()).sum()
                      for s in range(0, len(x), 500))
        one = ort_p.preprocess(x[:1]); batch = ort_p.preprocess(x[:64])
        ort_lat = latency_ms(lambda: ort_p.logits(one), args.runs)
        pt_lat = latency_ms(lambda: pt_p.logits(one), args.runs)
        thr = 64 / (latency_ms(lambda: ort_p.logits(batch), 10, 2) / 1000)
        mem = profile_onnx(os.path.join(ONNX_DIR, f"{name}.onnx"), ort_p.resolution, threads=args.threads,
                           runs=10)["peak_mem_mb"]
        mode = "nas" if name == "nas_best" else name.rsplit("_", 1)[1]
        rows.append({"model": name, "dataset": ort_p.dataset, "training": mode,
                     "params_M": count_params(pt_p.model) / 1e6,
                     "onnx_size_MB": os.path.getsize(os.path.join(ONNX_DIR, f"{name}.onnx")) / 2**20,
                     "test_acc_%": 100 * correct / len(x), "torch_latency_ms": pt_lat, "ort_latency_ms": ort_lat,
                     "ort_throughput_img_s": thr, "ort_peak_mem_MB": mem})
        print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in rows[-1].items()}, flush=True)

    df = pd.DataFrame(rows)
    # KD gain table (CIFAR-100 students)
    gains = []
    for s in ("mobilenetv2", "squeezenet", "efficientnet_lite0"):
        b, k = df[df.model == f"{s}_baseline"], df[df.model == f"{s}_kd"]
        if len(b) and len(k):
            gains.append({"student": s, "baseline_acc_%": b["test_acc_%"].item(), "kd_acc_%": k["test_acc_%"].item(),
                          "kd_gain_pts": k["test_acc_%"].item() - b["test_acc_%"].item(),
                          "teacher_acc_%": teacher_acc})
    df.to_csv(os.path.join(RESULTS_DIR, "benchmark.csv"), index=False)
    pd.DataFrame(gains).to_csv(os.path.join(RESULTS_DIR, "kd_vs_baseline.csv"), index=False)
    print("\n" + df.round(3).to_string(index=False))
    if gains:
        print("\n" + pd.DataFrame(gains).round(2).to_string(index=False))
    nb = os.path.join(RESULTS_DIR, "nas_best_arch.json")
    if os.path.exists(nb):
        print("\nNAS constraints:", load_json(nb)["constraints"])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/edge-dist/scripts/make_report.py
"""Build figures + the technical report PDF (docs/technical_report.pdf) from
the results/ and logs/ produced by the pipeline.

    python scripts/make_report.py
"""
import json
import os
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402
from reportlab.lib import colors  # noqa: E402
from reportlab.lib.pagesizes import A4  # noqa: E402
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet  # noqa: E402
from reportlab.lib.units import cm  # noqa: E402
from reportlab.platypus import (Image, PageBreak, Paragraph, Preformatted, SimpleDocTemplate,  # noqa: E402
                                Spacer, Table, TableStyle)

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)
from edgekd.utils import RESULTS_DIR  # noqa: E402

DOCS = os.path.join(ROOT, "docs"); FIG = os.path.join(DOCS, "figures"); LOGS = os.path.join(ROOT, "logs")
os.makedirs(FIG, exist_ok=True)

# reference categorical palette (light mode) + text/grid tokens
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df", "#fcfcfb"
STUDENTS = ["mobilenetv2", "squeezenet", "efficientnet_lite0"]
PRETTY = {"mobilenetv2": "MobileNetV2", "squeezenet": "SqueezeNet", "efficientnet_lite0": "EfficientNet-Lite0"}

plt.rcParams.update({
    "font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.titlesize": 10, "axes.titlecolor": INK, "legend.frameon": False,
})


def rj(name):
    p = os.path.join(RESULTS_DIR, name)
    return json.load(open(p)) if os.path.exists(p) else None


def rcsv(name):
    p = os.path.join(RESULTS_DIR, name)
    return pd.read_csv(p) if os.path.exists(p) else None


# ----------------------------------------------------------------------------- figures
def fig_kd_vs_baseline(kd):
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    x = range(len(kd)); w = 0.36
    b1 = ax.bar([i - w / 2 - 0.01 for i in x], kd["baseline_acc_%"], w, color=BLUE, label="Baseline (no KD)")
    b2 = ax.bar([i + w / 2 + 0.01 for i in x], kd["kd_acc_%"], w, color=ORANGE, label="Distilled (KD)")
    for bars in (b1, b2):
        for r in bars:
            ax.text(r.get_x() + r.get_width() / 2, r.get_height() + 0.6, f"{r.get_height():.1f}",
                    ha="center", va="bottom", fontsize=8, color=INK)
    t = kd["teacher_acc_%"].iloc[0]
    if pd.notna(t):
        ax.axhline(t, color=MUTED, lw=1.2, ls="--")
        ax.text(len(kd) - 0.5, t + 0.6, f"ResNet-50 teacher {t:.1f}%", ha="right", va="bottom", fontsize=8, color=INK2)
    ax.set_xticks(list(x)); ax.set_xticklabels([PRETTY[s] for s in kd["student"]])
    ax.set_ylabel("CIFAR-100 test accuracy (%)"); ax.grid(axis="x", visible=False)
    ax.set_ylim(0, max(kd["kd_acc_%"].max(), t if pd.notna(t) else 0) * 1.18)
    ax.legend(loc="upper left", ncol=2)
    ax.set_title("Knowledge distillation vs. baseline training (same budget)", loc="left")
    fig.tight_layout(); p = os.path.join(FIG, "kd_vs_baseline.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p


def fig_curves():
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.5), sharey=True)
    for ax, s in zip(axes, STUDENTS):
        for mode, c, lab in (("baseline", BLUE, "Baseline"), ("kd", ORANGE, "KD")):
            h = rj(f"{s}_{mode}_history.json")
            if h:
                e = [r["epoch"] for r in h["history"]]; a = [r["test_acc"] for r in h["history"]]
                ax.plot(e, a, color=c, lw=2, marker="o", ms=4, label=lab)
        ax.set_title(PRETTY[s], loc="left"); ax.set_xlabel("epoch")
    axes[0].set_ylabel("test acc (%)"); axes[0].legend(loc="upper left")
    fig.tight_layout(); p = os.path.join(FIG, "training_curves.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p


def fig_nas(hist, cons, best):
    df = pd.DataFrame(hist)
    acc, rej = df[df.status == "accepted"], df[df.status == "rejected"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 3.0))
    a1.scatter(acc.latency_ms, acc.size_mb, s=28, color=BLUE, label=f"accepted ({len(acc)})",
               edgecolor=SURFACE, lw=1)
    a1.scatter(rej.latency_ms, rej.size_mb, s=34, color=ORANGE, marker="x", lw=1.6, label=f"rejected ({len(rej)})")
    a1.axvline(cons["max_latency_ms"], color=MUTED, ls="--", lw=1); a1.axhline(cons["max_size_mb"], color=MUTED,
                                                                            ls="--", lw=1)
    a1.scatter([best["latency_ms"]], [best["size_mb"]], s=120, facecolor="none", edgecolor=INK, lw=1.5)
    a1.annotate("best", (best["latency_ms"], best["size_mb"]), xytext=(8, 6), textcoords="offset points",
                fontsize=8, color=INK)
    a1.set_xscale("log"); a1.set_yscale("log")
    from matplotlib.ticker import FuncFormatter
    plain = FuncFormatter(lambda v, _: f"{v:g}")
    for axis in (a1.xaxis, a1.yaxis):
        axis.set_major_formatter(plain); axis.set_minor_formatter(FuncFormatter(lambda v, _: ""))
    a1.set_xlabel("CPU latency, batch 1 (ms, log)"); a1.set_ylabel("ONNX size (MB, log)")
    a1.set_title("Hardware profile of every candidate", loc="left"); a1.legend(loc="upper left", fontsize=8)

    a2.scatter(acc["id"], acc.val_acc, s=24, color=BLUE, edgecolor=SURFACE, lw=1, label="candidate")
    best_so_far = acc.val_acc.cummax()
    a2.step(acc["id"], best_so_far, where="post", color=ORANGE, lw=2, label="best so far")
    a2.set_xlabel("candidate # (search order)"); a2.set_ylabel("proxy val acc (%)")
    a2.set_title("Evolutionary search progress", loc="left"); a2.legend(loc="lower right", fontsize=8)
    fig.tight_layout(); p = os.path.join(FIG, "nas_search.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p


def fig_latency(bench):
    b = bench.sort_values("ort_latency_ms")
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    y = range(len(b)); h = 0.38
    ax.barh([i + h / 2 + 0.01 for i in y], b.torch_latency_ms, h, color=BLUE, label="PyTorch (eager)")
    ax.barh([i - h / 2 - 0.01 for i in y], b.ort_latency_ms, h, color=ORANGE, label="ONNX Runtime")
    for i, (t, o) in enumerate(zip(b.torch_latency_ms, b.ort_latency_ms)):
        ax.text(o + 0.05, i - h / 2, f"{o:.2f}", va="center", fontsize=7, color=INK)
    ax.set_yticks(list(y)); ax.set_yticklabels(b.model, fontsize=8)
    ax.set_xlabel("batch-1 CPU latency, 1 thread (ms)"); ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right"); ax.set_title("CPU inference latency", loc="left")
    fig.tight_layout(); p = os.path.join(FIG, "latency.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p


def log_capture(name, max_lines=28):
    """Render the tail of a terminal log as a screenshot-style image."""
    p = os.path.join(LOGS, name)
    if not os.path.exists(p):
        return None
    lines = [ln.rstrip() for ln in open(p, errors="ignore").read().replace("\r", "\n").splitlines()
             if ln.strip() and "%|" not in ln and "Warning" not in ln and "warn(" not in ln]
    lines = [ln[:118] for ln in lines[-max_lines:]]
    fig = plt.figure(figsize=(7.2, 0.17 * len(lines) + 0.5))
    fig.patch.set_facecolor("#1e1e1e")
    fig.text(0.015, 0.985, f"$ {name.replace('.log', '')}", color="#7fd16b", family="monospace", fontsize=7.5,
             va="top")
    fig.text(0.015, 0.985 - 0.30 / (0.17 * len(lines) + 0.5), "\n".join(lines), color="#e6e6e6",
             family="monospace", fontsize=6.5, va="top", linespacing=1.35)
    out = os.path.join(FIG, f"log_{name.replace('.log', '')}.png")
    fig.savefig(out, dpi=200, facecolor=fig.get_facecolor()); plt.close(fig)
    return out


# ----------------------------------------------------------------------------- PDF
def table(df, fmt=None, widths=None):
    data = [list(df.columns)] + [[(fmt or {}).get(c, "{}").format(v) if not isinstance(v, str) else v
                                  for c, v in row.items()] for _, row in df.iterrows()]
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("FONT", (0, 0), (-1, 0), "Helvetica-Bold", 7.5), ("FONT", (0, 1), (-1, -1), "Helvetica", 7.5),
        ("TEXTCOLOR", (0, 0), (-1, -1), colors.HexColor(INK)),
        ("LINEBELOW", (0, 0), (-1, 0), 0.8, colors.HexColor(INK2)),
        ("LINEBELOW", (0, 1), (-1, -1), 0.3, colors.HexColor(GRID)),
        ("ALIGN", (1, 0), (-1, -1), "RIGHT"), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3), ("TOPPADDING", (0, 0), (-1, -1), 3)]))
    return t


def img(path, width=17 * cm):
    from reportlab.lib.utils import ImageReader
    iw, ih = ImageReader(path).getSize()
    return Image(path, width=width, height=width * ih / iw)


def main():
    ss = getSampleStyleSheet()
    H1 = ParagraphStyle("h1", parent=ss["Heading1"], fontSize=15, spaceAfter=6, textColor=colors.HexColor(INK))
    H2 = ParagraphStyle("h2", parent=ss["Heading2"], fontSize=11.5, spaceBefore=8, spaceAfter=4,
                        textColor=colors.HexColor(INK))
    B = ParagraphStyle("b", parent=ss["BodyText"], fontSize=9, leading=12.5, textColor=colors.HexColor(INK))
    CAP = ParagraphStyle("cap", parent=B, fontSize=8, textColor=colors.HexColor(INK2), spaceAfter=8)
    CODE = ParagraphStyle("code", fontName="Courier", fontSize=7.5, leading=9.5)
    s = []
    P = lambda t, st=B: s.append(Paragraph(t, st))  # noqa: E731

    teacher = rj("teacher_history.json"); kd = rcsv("kd_vs_baseline.csv"); bench = rcsv("benchmark.csv")
    nas_h = rj("nas_history.json"); nas_best = rj("nas_best_arch.json"); nas_tr = rj("nas_best_train_history.json")
    import argparse
    from edgekd.config import PROFILES
    ap = argparse.ArgumentParser(); ap.add_argument("--profile", default="quick")
    prof = ap.parse_args().profile
    Q = PROFILES[prof]
    env = rj("env.json") or {}

    P("Edge AI: Knowledge Distillation and Hardware-Aware NAS", H1)
    P("Technical report &mdash; ResNet-50 &rarr; MobileNetV2 / SqueezeNet / EfficientNet-Lite0 distillation on "
      "CIFAR-100, and evolutionary hardware-aware neural architecture search on CIFAR-10 with ONNX Runtime "
      "deployment.", CAP)

    P("1. Overview and compute setting", H2)
    hw = (f"GPU: {env.get('gpu', 'none')}; CPU: {env.get('cpu', 'n/a')}; PyTorch {env.get('torch', '?')}, "
          f"ONNX Runtime {env.get('onnxruntime', '?')}") if env else "see README"
    P(f"Hardware: {hw}. Training ran on the GPU with mixed precision (fp16 autocast, channels-last); all latency "
      "and memory measurements use ONNX Runtime / PyTorch on the <b>CPU</b> (deployment target).")
    P(f"All numbers in this report use the <b>{prof}</b> training profile from "
      "<font face='Courier'>edgekd/config.py</font>, a deliberately short schedule (the whole pipeline runs in about "
      "half an hour). A <b>full</b> profile (30-epoch teacher at 224 px, 200-epoch students, larger NAS) is provided "
      "for converged results. Absolute accuracies are therefore below published CIFAR numbers; the experiments "
      "demonstrate every component end to end and give a like-for-like comparison (identical budgets, seeds and data "
      "order for baseline and KD runs).")

    P("2. Question 1 &mdash; Knowledge distillation framework", H2)
    P("<b>Teacher.</b> torchvision ResNet-50 initialised from ImageNet weights, final FC replaced by a 100-way "
      f"classifier. CIFAR images are bilinearly upsampled to {Q['teacher']['input_res']} px inside the model. "
      + ("The stem, layer1 and layer2 are frozen (weights and BN statistics); layer3, layer4 and FC are fine-tuned"
         if Q["teacher"]["freeze"] else "All layers are fine-tuned")
      + f" for {Q['teacher']['epochs']} epochs on {Q['teacher']['train_subset'] or 50000} images "
      "(SGD + Nesterov, one-cycle LR, label smoothing 0.1).")
    P("<b>Students</b> (<font face='Courier'>edgekd/models/students.py</font>, implemented from scratch, stem stride "
      "set to 1 for 32 px inputs): <b>MobileNetV2</b> (inverted residuals, linear bottlenecks, width 1.0, 1280-d head); "
      "<b>SqueezeNet 1.1</b> (Fire modules with squeeze 1x1 / expand 1x1+3x3, conv-classifier + global pooling, "
      "BatchNorm added for stable short training); <b>EfficientNet-Lite0</b> (MBConv stages of B0 with kernels 3/5, "
      "no squeeze-excite, ReLU6, fixed stem/head &mdash; the mobile/edge-TPU variant).")
    P("<b>Distillation</b> (<font face='Courier'>edgekd/kd/losses.py</font>): temperature-scaled softmax "
      "p<sub>i</sub> = exp(z<sub>i</sub>/T) / &Sigma;<sub>j</sub> exp(z<sub>j</sub>/T); soft-target loss "
      "T&sup2;&middot;KL(p<sup>T</sup><sub>teacher</sub> || p<sup>T</sup><sub>student</sub>); combined objective "
      "L = (1&minus;&alpha;)&middot;CE(y, s) + &alpha;&middot;T&sup2;&middot;KL + &beta;&middot;L<sub>feat</sub>. "
      "Feature distillation (FitNets-style hint): the student's last feature map is projected to the teacher's 2048 "
      "channels by a learnable 1x1 conv + BN regressor, both maps are pooled to 2x2, L2-normalised per sample and "
      "matched with MSE. The regressor is trained jointly and discarded at deployment. "
      f"Hyper-parameters: T = {Q['kd']['temperature']}, &alpha; = {Q['kd']['alpha']}, &beta; = {Q['kd']['beta']}. "
      "The teacher is frozen and runs in eval mode on the same augmented batch as the student.")
    P(f"<b>Baselines</b> use the identical student code, budget ({Q['student']['epochs']} epochs on "
      f"{Q['student']['train_subset'] or 50000} images, one-cycle LR {Q['student']['lr']}), seed and data order, "
      "trained with hard labels only.")

    if teacher:
        P(f"Teacher result: <b>{teacher['history'][-1]['test_acc']:.2f}%</b> CIFAR-100 test accuracy "
          f"({teacher['params'] / 1e6:.1f} M parameters).")
    if kd is not None and len(kd):
        s.append(img(fig_kd_vs_baseline(kd), 15 * cm))
        P("Figure 1. CIFAR-100 test accuracy of each student with and without distillation.", CAP)
        k2 = kd.copy(); k2["student"] = k2["student"].map(PRETTY)
        s.append(table(k2, {c: "{:.2f}" for c in k2.columns if c != "student"}))
        P("Table 1. KD vs. baseline (results/kd_vs_baseline.csv).", CAP)
        s.append(img(fig_curves(), 17 * cm))
        P("Figure 2. Test accuracy per epoch (baseline vs. KD).", CAP)

    s.append(PageBreak())
    P("3. Question 2 &mdash; Hardware-aware neural architecture search", H2)
    P("<b>Search space</b> (<font face='Courier'>edgekd/nas/search_space.py</font>): input resolution {24, 28, 32}; "
      "stem width {16, 24, 32}; 2&ndash;4 stages (each opens with a stride-2 block) of 1&ndash;3 blocks "
      "&rArr; variable depth; per-stage channels {16 &hellip; 128} (non-decreasing); kernel {3, 5, 7}; block type "
      "{standard conv, depthwise-separable conv, MBConv with expansion {2, 4, 6}}; activation {ReLU, ReLU6, "
      "Hardswish, SiLU, LeakyReLU}. Architectures are plain JSON dicts.")
    P("<b>Generator</b> (<font face='Courier'>edgekd/nas/generator.py</font>) turns a config into an "
      "<font face='Courier'>nn.Module</font> (stem &rarr; stages &rarr; global pooling &rarr; linear), adding residual "
      "connections whenever shapes match.")
    P("<b>Hardware-aware evaluation</b> (<font face='Courier'>edgekd/nas/hardware.py</font>): each candidate is "
      "exported to ONNX and profiled in a fresh subprocess with ONNX Runtime on the CPU, batch 1, one intra-op "
      "thread (mimicking a small edge core): median latency over 50 runs after 10 warm-ups, storage size of the "
      ".onnx file, and peak memory (peak-RSS increase after resetting the kernel high-water mark via "
      "/proc/self/clear_refs; reported as unsupported on non-Linux). Candidates violating the deployment "
      "constraints are <b>rejected before any training</b>.")
    P("<b>Search strategy</b> (<font face='Courier'>edgekd/nas/evolution.py</font>): (&mu;+&lambda;) evolutionary "
      "search with tournament selection (k = 3), uniform stage-wise crossover and 1&ndash;2-gene mutation "
      "(including adding/removing a stage), de-duplication cache and elitist survival. Fitness of deployable "
      f"candidates is validation accuracy after {Q['nas']['proxy_epochs']} proxy epochs on "
      f"{Q['nas']['proxy_train']} CIFAR-10 training images, evaluated on a held-out split of "
      f"{Q['nas']['val']} training images (the test set is never used for selection). Population "
      f"{Q['nas']['population']}, {Q['nas']['generations']} generations. The best architecture is saved to "
      "results/nas_best_arch.json and re-trained from scratch on the full CIFAR-10 training set "
      f"({Q['nas']['final_epochs']} epochs).")
    if nas_h and nas_best:
        c = nas_h["constraints"]; H = nas_h["history"]
        n_rej = sum(r["status"] == "rejected" for r in H)
        P(f"Constraints: latency &le; {c['max_latency_ms']} ms, size &le; {c['max_size_mb']} MB, peak memory &le; "
          f"{c['max_peak_mem_mb']} MB. The search evaluated <b>{len(H)}</b> candidates in "
          f"{nas_h['search_time_s'] / 60:.1f} min; <b>{n_rej}</b> were rejected by the constraints.")
        s.append(img(fig_nas(H, c, nas_best), 17 * cm))
        P("Figure 3. Left: latency vs. size of every candidate with constraint limits (dashed). "
          "Right: proxy validation accuracy of deployable candidates and best-so-far.", CAP)
        a = nas_best["arch"]
        rows = [{"stage": i + 1, **{k: str(v) for k, v in st.items()}} for i, st in enumerate(a["stages"])]
        P(f"<b>Best architecture</b>: input {a['resolution']} px, stem {a['stem_channels']} ch, "
          f"{nas_best['params'] / 1e3:.1f} K params, {nas_best['latency_ms']:.2f} ms, {nas_best['size_mb']:.2f} MB, "
          f"peak mem {nas_best['peak_mem_mb']:.1f} MB, proxy val acc {nas_best['proxy_val_acc']:.2f}%.")
        rdf = pd.DataFrame(rows)
        if "expand" in rdf:
            rdf.loc[rdf.block != "mbconv", "expand"] = "-"
        s.append(table(rdf))
        P("Table 2. Best discovered architecture (results/nas_best_arch.json).", CAP)
        top = pd.DataFrame([r for r in H if r["status"] == "accepted"]).sort_values("val_acc", ascending=False)
        top = top.head(8)[["id", "generation", "params", "latency_ms", "size_mb", "peak_mem_mb", "val_acc"]]
        s.append(table(top, {"params": "{:.0f}", "latency_ms": "{:.2f}", "size_mb": "{:.3f}",
                             "peak_mem_mb": "{:.1f}", "val_acc": "{:.2f}"}))
        P("Table 3. Top deployable candidates (full log: results/nas_history.csv / .json).", CAP)
    if nas_tr:
        P(f"Re-trained from scratch: <b>{nas_tr['history'][-1]['test_acc']:.2f}%</b> CIFAR-10 test accuracy "
          f"after {len(nas_tr['history'])} epochs.")

    s.append(PageBreak())
    P("4. Deployment: ONNX export, unified inference, benchmarking", H2)
    P("<font face='Courier'>scripts/export_onnx.py</font> exports all six students and the NAS model (opset 17, "
      "dynamic batch), runs the ONNX checker and verifies ONNX Runtime outputs against PyTorch "
      "(max abs difference &lt; 1e-3). <font face='Courier'>edgekd/inference.py</font> provides one "
      "<font face='Courier'>Predictor</font> class for every model with interchangeable PyTorch / ONNX Runtime "
      "back-ends, including preprocessing (resize to model resolution, dataset normalisation) and top-k "
      "decoding; <font face='Courier'>scripts/infer.py</font> is its CLI. <font face='Courier'>scripts/benchmark.py</font> "
      "measures accuracy (ONNX Runtime, full test set), batch-1 single-thread latency for both back-ends, batch-64 "
      "throughput and peak memory, writing results/benchmark.csv.")
    if bench is not None and len(bench):
        b2 = bench[["model", "params_M", "onnx_size_MB", "test_acc_%", "torch_latency_ms", "ort_latency_ms",
                    "ort_throughput_img_s", "ort_peak_mem_MB"]]
        b2.columns = ["model", "params (M)", "ONNX MB", "acc %", "torch ms", "ORT ms", "ORT img/s", "peak MB"]
        s.append(table(b2, {"params (M)": "{:.3f}", "ONNX MB": "{:.2f}", "acc %": "{:.2f}", "torch ms": "{:.2f}",
                            "ORT ms": "{:.2f}", "ORT img/s": "{:.0f}", "peak MB": "{:.1f}"}))
        P("Table 4. Benchmark (students: CIFAR-100; nas_best: CIFAR-10). Latency: batch 1, 1 CPU thread.", CAP)
        s.append(img(fig_latency(bench), 15 * cm))
        P("Figure 4. Batch-1 CPU latency, PyTorch eager vs. ONNX Runtime.", CAP)

    P("5. Discussion and limitations", H2)
    if kd is not None and len(kd):
        g = kd["kd_gain_pts"]
        P(f"Under an identical, very short budget distillation changed student accuracy by "
          f"{', '.join(f'{PRETTY[s]} {v:+.2f}' for s, v in zip(kd.student, g))} points.")
        if (g < 0).any():
            P("<b>Why distillation hurt here.</b> Two diagnostics locate the cause. (i) The teacher, fine-tuned for "
              "only a few epochs, produces soft logits: at T = 4 its mean top-1 probability is 0.05 (0.74 at T = 1; "
              "uniform is 0.01), so the soft targets carry little class information while the hard-label term is "
              f"down-weighted to 1 &minus; &alpha; = {1 - Q['kd']['alpha']:.1f}. The KD training logs confirm it: the "
              "hard-label training loss of every KD student stays far above its baseline counterpart. (ii) The "
              f"feature-hint term (&beta; = {Q['kd']['beta']}) dominates the gradient. A controlled pilot (SqueezeNet, "
              "10k CIFAR-100 images, 3 epochs, same seed and data order, CPU) isolates it:")
            s.append(table(pd.DataFrame({
                "setting": ["baseline (hard labels)", "KD, T=4, alpha=0.7, beta=1 (used above)",
                            "KD, T=4, alpha=0.7, beta=0 (logit-only)"],
                "test acc %": [11.20, 3.55, 11.80]}), {"test acc %": "{:.2f}"}))
            P("Table 3b. KD ablation pilot (2,000 test images).", CAP)
            P("With the hint loss removed, logit distillation beats the baseline (+0.6 points) even with this soft "
              "teacher; with &beta; = 1 accuracy collapses. The recommended setting is therefore &beta; &le; 0.1 "
              "(or a hint loss applied only in a short warm-up stage), a longer-trained teacher and a lower "
              "temperature (T = 2 gives top-1 probability 0.23). Re-run with "
              "<font face='Courier'>train_students.py --mode kd --beta 0.1</font>.")
        else:
            P("Soft targets carry the teacher's inter-class similarity structure, which is a denser training signal "
              "than one-hot labels; this matters most under a short training budget, as here.")
        P("The students are not converged and results come from a single seed; the full profile should be run to "
          "obtain converged numbers.")
    P("The NAS rejected oversized/slow networks before spending any training compute on them, so the search budget "
      "is spent only inside the deployable region. Proxy accuracy after a short training run is a noisy, "
      "low-fidelity estimate of final accuracy; ranking correlation improves with longer proxy training "
      "(full profile). Latency is measured on the development CPU with one thread; for a different target device "
      "the same pipeline should be executed on that device (the profiler is a standalone script).")

    s.append(PageBreak())
    P("Appendix &mdash; terminal output captures", H2)
    for name in ("teacher.log", "students_baseline.log", "students_kd.log", "nas_search.log",
                 "nas_train_best.log", "export_onnx.log", "infer_onnx.log", "benchmark.log"):
        p = log_capture(name)
        if p:
            s.append(img(p, 17 * cm)); P(f"logs/{name}", CAP)
    P("Reproduce: see README.md (<font face='Courier'>./run_all.sh quick</font>).", B)

    out = os.path.join(DOCS, "technical_report.pdf")
    SimpleDocTemplate(out, pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=1.8 * cm,
                      bottomMargin=1.8 * cm, title="Edge AI: KD and Hardware-Aware NAS").build(s)
    print("wrote", out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/edge-dist/requirements.txt
--extra-index-url https://download.pytorch.org/whl/cpu
torch==2.15.0
torchvision==0.29.2
onnx==1.23.2
onnxruntime==1.32.0
onnxscript
numpy
pandas
matplotlib
psutil
pillow
reportlab

In [ ]:
%%writefile /content/edge-dist/README.md
# Edge AI: Knowledge Distillation + Hardware-Aware NAS

* **Q1:** ResNet-50 teacher, distilled into MobileNetV2, SqueezeNet and EfficientNet-Lite0 on CIFAR-100. Uses soft targets with temperature, combined hard and soft loss, and feature distillation. Includes baselines without KD, ONNX export and ONNX Runtime CPU inference.
* **Q2:** Evolutionary, hardware-aware NAS on CIFAR-10. Candidates are rejected on measured CPU latency, ONNX size and peak memory. The best architecture is retrained from scratch, then exported to ONNX for inference.

## Installation
```bash
conda create -y -p ./.venv python=3.11
conda activate ./.venv
pip install -r requirements.txt
# CIFAR-10/100 (python version) into ./data
mkdir -p data && cd data
curl -LO https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz && tar xzf cifar-100-python.tar.gz
curl -LO https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz  && tar xzf cifar-10-python.tar.gz
cd ..
```

## Run everything
```bash
./run_all.sh quick     # ~1 h on a 4-core laptop CPU (budgets used for the reported results)
./run_all.sh full      # conventional GPU-scale schedule
```
Budgets live in `edgekd/config.py`. The NAS deployment constraints are there too: 3 ms latency, 2 MB size, 25 MB peak memory.

## Step by step
| Step | Command | Output |
|---|---|---|
| 1. Teacher | `python scripts/train_teacher.py` | `checkpoints/teacher_resnet50.pt` |
| 2. Baseline students | `python scripts/train_students.py --mode baseline` | `checkpoints/*_baseline.pt` |
| 3. KD students | `python scripts/train_students.py --mode kd [--temperature 4 --alpha 0.7 --beta 1]` | `checkpoints/*_kd.pt` |
| 4. NAS search | `python scripts/nas_search.py [--max-latency-ms 3 --max-size-mb 2 --max-mem-mb 25]` | `results/nas_history.{csv,json}`, `results/nas_best_arch.json` |
| 5. Train best NAS model | `python scripts/nas_train_best.py` | `checkpoints/nas_best.pt` |
| 6. ONNX export | `python scripts/export_onnx.py` | `onnx_models/*.onnx` (checked against PyTorch) |
| 7. Inference | `python scripts/infer.py --model all --backend onnx --eval`<br>`python scripts/infer.py --model mobilenetv2_kd --image img.png` | predictions / accuracy |
| 8. Benchmark | `python scripts/benchmark.py` | `results/benchmark.csv`, `results/kd_vs_baseline.csv` |
| KD ablation pilot | `python scripts/kd_pilot.py` | Table 3b of the report (`logs/kd_pilot.log`) |
| 9. Report | `python scripts/make_report.py --profile colab` | `docs/technical_report.pdf`, `docs/figures/` |

The model names for `infer.py` are `{mobilenetv2,squeezenet,efficientnet_lite0}_{kd,baseline}` and `nas_best`. Use `--backend torch` to run with PyTorch instead of ONNX Runtime.

## Results (Colab T4, `colab` profile)
Full report: [`docs/technical_report.pdf`](docs/technical_report.pdf). Raw numbers: `results/benchmark.csv`, `results/kd_vs_baseline.csv`.

| model | params (M) | ONNX (MB) | test acc % | ORT CPU latency (ms, batch 1) |
|---|---|---|---|---|
| ResNet-50 teacher | 23.7 | - | 81.14 | - |
| MobileNetV2 baseline / KD | 2.35 | 8.9 | 54.44 / 30.69 | 1.15 / 1.68 |
| SqueezeNet 1.1 baseline / KD | 0.78 | 3.0 | 51.39 / 38.25 | 0.63 / 0.87 |
| EfficientNet-Lite0 baseline / KD | 3.50 | 13.3 | 56.67 / 48.80 | 1.47 / 2.04 |
| NAS best (CIFAR-10) | 0.14 | 0.53 | 88.18 | 0.53 |

With the configured feature-hint weight (beta = 1) distillation lowered accuracy in this short schedule. The report
(Section 5, Table 3b) traces this to the hint loss dominating the gradient and to the soft logits of the briefly
fine-tuned teacher; a controlled pilot (`scripts/kd_pilot.py`) shows logit-only KD beating the baseline. Use
`--beta 0.1` or `--beta 0` for future runs.

## Layout
```
edgekd/
  data.py            fast in-memory CIFAR loaders + tensor augmentation
  models/teacher.py  ResNet-50 teacher (ImageNet init, feature output)
  models/students.py MobileNetV2, SqueezeNet 1.1, EfficientNet-Lite0 (from scratch)
  kd/losses.py       temperature softmax, soft-target KL, combined loss, feature distiller
  kd/trainer.py      distillation training loop
  nas/search_space.py   search space, sampling, mutation, crossover
  nas/generator.py      config -> nn.Module
  nas/hardware.py       ONNX export, CPU latency / size / peak-memory profiler, constraint check
  nas/evolution.py      (mu+lambda) evolutionary search
  inference.py       unified Predictor (PyTorch / ONNX Runtime)
scripts/             train / search / export / infer / benchmark / report entry points
```

## Notes
The reported results come from a CPU-only machine with no CUDA GPU. To fit that, the quick profile fine-tunes an ImageNet-pretrained teacher at 64 px with stem through layer2 frozen. It trains students for 4 epochs on 25k images and uses short NAS proxy training. Accuracies are therefore far below converged values. Baseline and KD runs share an identical budget, seed and data order, so the two are directly comparable.

## Google Colab (T4 GPU)
`edge_kd_nas_colab.ipynb` is a single, self-contained notebook. It writes this package into `/content`, downloads CIFAR, runs every stage with the `colab` profile (about 35–45 min on a T4) and downloads `edge_kd_nas_full.zip` at the end (checkpoints, ONNX models, results, logs, report, code). Regenerate it after code changes with `python scripts/build_colab.py`.
Training uses CUDA with mixed precision when available. Latency and memory are always measured on the CPU, the deployment target.

In [ ]:
%%writefile /content/edge-dist/run_all.sh
#!/usr/bin/env bash
# End-to-end pipeline. Usage: ./run_all.sh [quick|full]   (default: quick, ~1h on a 4-core CPU)
set -euo pipefail
cd "$(dirname "$0")"
PROFILE="${1:-quick}"
PY="${PYTHON:-python}"
mkdir -p logs

echo "### Q1: knowledge distillation (CIFAR-100)"
$PY scripts/train_teacher.py  --profile "$PROFILE"            2>&1 | tee logs/teacher.log
$PY scripts/train_students.py --profile "$PROFILE" --mode baseline 2>&1 | tee logs/students_baseline.log
$PY scripts/train_students.py --profile "$PROFILE" --mode kd       2>&1 | tee logs/students_kd.log

echo "### Q2: hardware-aware NAS (CIFAR-10)"
$PY scripts/nas_search.py     --profile "$PROFILE"            2>&1 | tee logs/nas_search.log
$PY scripts/nas_train_best.py --profile "$PROFILE"            2>&1 | tee logs/nas_train_best.log

echo "### Deployment: ONNX export, inference, benchmark"
$PY scripts/export_onnx.py                                    2>&1 | tee logs/export_onnx.log
$PY scripts/infer.py --model all --backend onnx --eval        2>&1 | tee logs/infer_onnx.log
$PY scripts/benchmark.py                                      2>&1 | tee logs/benchmark.log
$PY scripts/make_report.py                                    2>&1 | tee logs/report.log

## 3. Data + environment info

In [ ]:
%cd /content/edge-dist
%env PYTHONUNBUFFERED=1
%env PYTHONWARNINGS=ignore
from edgekd.data import ensure_data
for _d in ("cifar100", "cifar10"):
    ensure_data(_d)  # download + md5-verify + extract; raises (stops Run all) on failure
!ls data
import json, os, platform, subprocess, torch, onnxruntime
os.makedirs("results", exist_ok=True)
cpu = subprocess.run("lscpu | grep 'Model name' | cut -d: -f2", shell=True, capture_output=True, text=True).stdout.strip()
env = {"gpu": torch.cuda.get_device_name(0), "cpu": f"{cpu} ({os.cpu_count()} vCPU)", "torch": torch.__version__,
       "onnxruntime": onnxruntime.__version__, "python": platform.python_version(), "profile": "colab"}
json.dump(env, open("results/env.json", "w"), indent=2); print(env)

## 4. Q1: Teacher (ResNet-50, CIFAR-100)

In [ ]:
%%bash
set -eo pipefail
cd /content/edge-dist
python scripts/train_teacher.py --profile colab 2>&1 | tee logs/teacher.log

## 5. Q1: Baseline students (no distillation)

In [ ]:
%%bash
set -eo pipefail
cd /content/edge-dist
python scripts/train_students.py --profile colab --mode baseline 2>&1 | tee logs/students_baseline.log

## 6. Q1: Distilled students (soft targets + temperature + hard labels + feature distillation)

In [ ]:
%%bash
set -eo pipefail
cd /content/edge-dist
python scripts/train_students.py --profile colab --mode kd 2>&1 | tee logs/students_kd.log

## 7. Q2: Hardware-aware evolutionary NAS (CIFAR-10)
Latency and peak memory are measured on the Colab **CPU**: ONNX Runtime, batch 1, one thread. Candidates over the limits are rejected before any training.

In [ ]:
%%bash
set -eo pipefail
cd /content/edge-dist
python scripts/nas_search.py --profile colab --max-latency-ms 4 --max-size-mb 2 --max-mem-mb 30 2>&1 | tee logs/nas_search.log

## 8. Q2: Train the best discovered architecture from scratch

In [ ]:
%%bash
set -eo pipefail
cd /content/edge-dist
python scripts/nas_train_best.py --profile colab 2>&1 | tee logs/nas_train_best.log

## 9. ONNX export + ONNX Runtime CPU inference

In [ ]:
%%bash
set -eo pipefail
cd /content/edge-dist
python scripts/export_onnx.py 2>&1 | tee logs/export_onnx.log
python scripts/infer.py --model all --backend onnx --eval 2>&1 | tee logs/infer_onnx.log
python scripts/infer.py --model mobilenetv2_kd --backend onnx --samples 5 2>&1 | tee -a logs/infer_onnx.log
python scripts/infer.py --model nas_best --backend onnx --samples 5 2>&1 | tee -a logs/infer_onnx.log

## 10. Benchmark (CPU) + technical report

In [ ]:
%%bash
set -eo pipefail
cd /content/edge-dist
python scripts/benchmark.py 2>&1 | tee logs/benchmark.log
python scripts/make_report.py --profile colab 2>&1 | tee logs/report.log

In [ ]:
from IPython.display import Image, display
for f in ["kd_vs_baseline", "training_curves", "nas_search", "latency"]:
    display(Image(f"docs/figures/{f}.png", width=800))

## 11. Download everything
`edge_kd_nas_full.zip` is the complete project, ready to push to GitHub:
* `checkpoints/`: teacher, 3 baseline, 3 KD and NAS `.pt` files
* `onnx_models/`: 7 `.onnx` models
* `results/`: CSV/JSON results, including the NAS history and best architecture
* `logs/`, `docs/` (PDF report and figures), plus the source code, README and requirements

CIFAR data is excluded. Allow the download if the browser asks.

In [ ]:
import os
from google.colab import files
!rm -f edge_kd_nas_full.zip && zip -qr edge_kd_nas_full.zip . -x "data/*" "*__pycache__*" "*.zip"
!unzip -l edge_kd_nas_full.zip | grep -E "\.(pt|onnx|csv|json|pdf)$"
!ls -lh edge_kd_nas_full.zip
files.download("edge_kd_nas_full.zip")